# Distinct, overlapping, and redundant dimensions of recording-quality variation

**Manuscript Goal 2.** This production notebook separates persistent between-participant structure from recording-to-recording within-participant co-variation. It estimates pairwise structure, evaluates family organization, classifies overlap and potential redundancy, and performs multilevel PCA without constructing a family or global quality score.

The notebook is governed by `Goal2_Frozen_Multidimensional_Structure_Plan.docx`. Any change to a support gate, feature set, transformation, bootstrap count, or interpretation rule requires a versioned amendment before affected results are reviewed.


## 0. Frozen statistical specification

Primary between-participant structure is the Spearman correlation of one transformed mean profile per participant. Primary within-participant structure is a rank-based correlation of participant-centered recording values with equal total weight per participant. The two matrices are estimated and interpreted separately.

Pairwise analyses include every retained nonconstant QADD, QGAIN, QREV, and QCHAN output that passes pair-specific support gates. QDIST and QTEMP are audited but are excluded when sparse or invariant. Primary multilevel PCA uses a prespecified eight-feature complete-case set chosen for complementary family coverage, empirical support, and avoidance of known ledger dependence.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import re
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Sequence

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import stats
from scipy.optimize import linear_sum_assignment
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.cov_struct import Exchangeable, Independence
from statsmodels.stats.multitest import multipletests
from IPython.display import display, Markdown

warnings.filterwarnings('default')
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

# ------------------------------------------------------------------
# Run configuration
# ------------------------------------------------------------------
FINAL_RUN = True
SEED = 20260805

N_CORRELATION_BOOTSTRAP = 2000 if FINAL_RUN else 30
N_FAMILY_BOOTSTRAP = 2000 if FINAL_RUN else 30
N_FAMILY_PERMUTATIONS = 10000 if FINAL_RUN else 500
N_PCA_BOOTSTRAP = 2000 if FINAL_RUN else 100
N_PARALLEL_ANALYSIS = 1000 if FINAL_RUN else 100
N_RANDOM_PAIR_RESAMPLES = 1000 if FINAL_RUN else 30

MIN_BETWEEN_PARTICIPANTS = 50
MIN_WITHIN_REPEATED_PARTICIPANTS = 20
MIN_WITHIN_JOINT_RECORDINGS = 50
MIN_BOOTSTRAP_SUCCESS = 0.90
MIN_SPARSE_POSITIVE_RECORDINGS = 20
MIN_SPARSE_POSITIVE_PARTICIPANTS = 10
PCA_MIN_COMPLETE_PARTICIPANTS = 100
PCA_MIN_REPEATED_PARTICIPANTS = 50
PCA_LOADING_STABILITY_MEDIAN = 0.80
PCA_LOADING_STABILITY_P05 = 0.50

rng = np.random.default_rng(SEED)

# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------
def locate_analysis_root() -> Path:
    configured = os.environ.get("PAPER1_ANALYSIS_ROOT", "").strip()
    if configured:
        return Path(configured).expanduser().resolve()
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "requirements.txt").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the analysis repository root")

ANALYSIS_ROOT = locate_analysis_root()
DEFAULT_PAPER_ROOT = ANALYSIS_ROOT.parents[1]
PAPER_ROOT = Path(os.environ.get("PAPER1_ROOT", DEFAULT_PAPER_ROOT))
PIPELINE_ROOT = Path(os.environ.get("PAPER1_PIPELINE_ROOT", PAPER_ROOT / "paper_1_cleanup"))

DATA_FREEZE_DIR = Path(os.environ.get(
    'PAPER1_DATA_FREEZE_DIR',
    PIPELINE_ROOT / 'MAIN outputs' / '00_DATA_FREEZE' / 'v1',
))
FAMILY_FREEZE_DIR = Path(os.environ.get(
    'PAPER1_FAMILY_FREEZE_DIR',
    PIPELINE_ROOT / 'MAIN outputs' / '02_FEATURE_REVIEWED' / '06_family_freezes',
))

OUTPUT_ROOT = ANALYSIS_ROOT / 'outputs' / 'multidimensional_structure'
TABLE_DIR = OUTPUT_ROOT / 'tables'
MODEL_DIR = OUTPUT_ROOT / 'models'
LOG_DIR = OUTPUT_ROOT / 'logs'
MANIFEST_DIR = OUTPUT_ROOT / 'manifests'
FIGURE_MAIN_DIR = OUTPUT_ROOT / 'figures' / 'main'
FIGURE_SUPP_DIR = OUTPUT_ROOT / 'figures' / 'supplementary'
FIGURE_SOURCE_DIR = OUTPUT_ROOT / 'figures' / 'source_data'
BOOTSTRAP_DIR = MODEL_DIR / 'bootstrap_checkpoints'

for directory in [
    TABLE_DIR, MODEL_DIR, LOG_DIR, MANIFEST_DIR,
    FIGURE_MAIN_DIR, FIGURE_SUPP_DIR, FIGURE_SOURCE_DIR,
    BOOTSTRAP_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

FROZEN_SPEC = {
    'analysis_name': 'Distinct, overlapping, and redundant multidimensional structure',
    'primary_between_estimand': 'Spearman correlation of participant mean profiles',
    'primary_within_estimand': 'rank-based participant-centered correlation with equal participant weight',
    'between_gate_participants': MIN_BETWEEN_PARTICIPANTS,
    'within_gate_repeated_participants': MIN_WITHIN_REPEATED_PARTICIPANTS,
    'within_gate_joint_recordings': MIN_WITHIN_JOINT_RECORDINGS,
    'correlation_bootstrap': N_CORRELATION_BOOTSTRAP,
    'family_bootstrap': N_FAMILY_BOOTSTRAP,
    'family_permutations': N_FAMILY_PERMUTATIONS,
    'pca_bootstrap': N_PCA_BOOTSTRAP,
    'parallel_analysis': N_PARALLEL_ANALYSIS,
    'seed': SEED,
    'final_run': FINAL_RUN,
    'no_imputation': True,
    'no_family_or_global_score': True,
}
(MANIFEST_DIR / 'frozen_analysis_spec.json').write_text(
    json.dumps(FROZEN_SPEC, indent=2), encoding='utf-8'
)

print(json.dumps(FROZEN_SPEC, indent=2))
print(f'DATA_FREEZE_DIR:   {DATA_FREEZE_DIR}')
print(f'FAMILY_FREEZE_DIR: {FAMILY_FREEZE_DIR}')
print(f'OUTPUT_ROOT:       {OUTPUT_ROOT}')


## 1. Shared utilities


In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path, low_memory=False)
    if suffix == '.parquet':
        try:
            return pd.read_parquet(path)
        except ImportError:
            csv_fallback = path.with_suffix('.csv')
            if csv_fallback.exists():
                warnings.warn(
                    f'Parquet engine unavailable; using CSV fallback: {csv_fallback}',
                    RuntimeWarning,
                )
                return pd.read_csv(csv_fallback, low_memory=False)
            raise
    raise ValueError(f'Unsupported table type: {path}')


def resolve_table(
    root: Path,
    exact_relatives: Sequence[str],
    patterns: Sequence[str],
    *,
    label: str,
) -> Path:
    for relative in exact_relatives:
        candidate = root / relative
        if candidate.exists():
            return candidate

    matches: list[Path] = []
    for pattern in patterns:
        matches.extend(path for path in root.rglob(pattern) if path.is_file())

    matches = sorted(
        {path.resolve() for path in matches},
        key=lambda path: (
            0 if path.suffix.lower() == '.parquet' else 1,
            'source_candidate' in str(path).lower(),
            len(str(path)),
            str(path).lower(),
        ),
    )
    preferred = [
        path for path in matches
        if 'checkpoint' not in str(path).lower()
    ]
    if preferred:
        matches = preferred
    if not matches:
        raise FileNotFoundError(
            f'Could not locate {label} below {root}. Patterns: {patterns}'
        )
    return matches[0]


def safe_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors='coerce')


def normalize_id(series: pd.Series) -> pd.Series:
    return series.astype('string').str.strip()


def transform_feature(series: pd.Series, transform: str) -> pd.Series:
    x = safe_numeric(series)
    if transform == 'none':
        return x
    if transform == 'log1p':
        if (x.dropna() < 0).any():
            raise ValueError('log1p requested for a feature containing negative values.')
        return np.log1p(x)
    if transform == 'asinh':
        return np.arcsinh(x)
    raise ValueError(f'Unknown transform: {transform}')


def robust_scale(
    series: pd.Series,
    *,
    median: float | None = None,
    iqr: float | None = None,
) -> tuple[pd.Series, float, float]:
    numeric = safe_numeric(series)
    if median is None:
        median = float(numeric.median(skipna=True))
    if iqr is None:
        q1, q3 = numeric.quantile([0.25, 0.75])
        iqr = float(q3 - q1)
    if not np.isfinite(iqr) or iqr <= 0:
        return pd.Series(np.nan, index=series.index, dtype=float), median, iqr
    return (numeric - median) / iqr, median, iqr


def bh_adjust(frame: pd.DataFrame, p_col: str = 'p_value') -> pd.DataFrame:
    output = frame.copy()
    output['q_value'] = np.nan
    mask = output[p_col].notna() & np.isfinite(output[p_col])
    if mask.any():
        output.loc[mask, 'q_value'] = multipletests(
            output.loc[mask, p_col].astype(float), method='fdr_bh'
        )[1]
    return output


def percentile_interval(values: Iterable[float], alpha: float = 0.05) -> tuple[float, float]:
    clean = np.asarray([value for value in values if np.isfinite(value)], dtype=float)
    if clean.size == 0:
        return np.nan, np.nan
    return tuple(np.quantile(clean, [alpha / 2, 1 - alpha / 2]))


def export_dataframe(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def save_figure(fig: mpl.figure.Figure, stem: str, *, main: bool) -> dict[str, str]:
    """Export deterministic fixed-canvas publication graphics."""
    directory = FIGURE_MAIN_DIR if main else FIGURE_SUPP_DIR
    paths = {
        'png': directory / f'{stem}.png',
        'svg': directory / f'{stem}.svg',
    }
    # The editable SVG is the vector deliverable for all figures. The main figure
    # additionally receives a PDF. Supplementary PNGs use 350 dpi to keep the
    # complete production run practical without affecting vector quality.
    fig.savefig(paths['png'], dpi=600 if main else 350, facecolor='white')
    fig.savefig(paths['svg'], facecolor='white')
    if main:
        paths['pdf'] = directory / f'{stem}.pdf'
        fig.savefig(paths['pdf'], facecolor='white')
    return {key: str(value) for key, value in paths.items()}


def model_row(
    *, feature: str, analysis_name: str, target_term: str,
    estimate: float, standard_error: float, ci95_low: float, ci95_high: float,
    p_value: float, n_recordings: int, n_participants: int,
    model: str, converged: bool, random_intercept_variance: float = np.nan,
    notes: str = '',
) -> dict[str, Any]:
    return {
        'feature': feature,
        'analysis': analysis_name,
        'target_term': target_term,
        'estimate': estimate,
        'standard_error': standard_error,
        'ci95_low': ci95_low,
        'ci95_high': ci95_high,
        'p_value': p_value,
        'n_recordings': n_recordings,
        'n_participants': n_participants,
        'model': model,
        'converged': bool(converged),
        'random_intercept_variance': random_intercept_variance,
        'notes': notes,
    }


def bowley_skewness(values: pd.Series) -> float:
    clean = safe_numeric(values).dropna()
    if clean.empty:
        return np.nan
    q1, q2, q3 = clean.quantile([0.25, 0.50, 0.75])
    denominator = q3 - q1
    if not np.isfinite(denominator) or denominator == 0:
        return np.nan
    return float((q3 + q1 - 2 * q2) / denominator)


def boolean_series(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return (
        series.astype('string').str.strip().str.lower()
        .map({'true': True, 'false': False, '1': True, '0': False,
              'yes': True, 'no': False})
        .fillna(False)
        .astype(bool)
    )


## 2. Discover and hash immutable inputs


In [ ]:
metadata_path = resolve_table(
    DATA_FREEZE_DIR,
    exact_relatives=['bamboo_recording_freeze_ledger.csv', 'bamboo_recording_freeze_ledger.parquet'],
    patterns=['*recording*freeze*ledger*.parquet', '*recording*freeze*ledger*.csv'],
    label='recording freeze ledger',
)

family_paths = {
    'QADD': resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            r'additive_interference\qadd-v4.2.0\tables\qadd_v420_recording_features.parquet',
            r'additive_interference\qadd-v4.2.0\tables\qadd_v420_recording_features.csv',
        ],
        patterns=['qadd_v420_recording_features.parquet', 'qadd_v420_recording_features.csv'],
        label='QADD recording features',
    ),
    'QGAIN': resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            r'gain_dynamics\qgain-v4.1.0\tables\qgain_v410_recording_features.parquet',
            r'gain_dynamics\qgain-v4.1.0\tables\qgain_v410_recording_features.csv',
        ],
        patterns=['qgain_v410_recording_features.parquet', 'qgain_v410_recording_features.csv'],
        label='QGAIN recording features',
    ),
    'QREV': resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            r'reverberation\qrev-v4.0.0\tables\qrev_v400_analysis_features.parquet',
            r'reverberation\qrev-v4.0.0\tables\qrev_v400_analysis_features.csv',
        ],
        patterns=['qrev_v400_analysis_features.parquet', 'qrev_v400_analysis_features.csv'],
        label='QREV analysis features',
    ),
    'QCHAN': resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            r'channel_device\qchan-v4.0.0\tables\qchan_v400_analysis_features.parquet',
            r'channel_device\qchan-v4.0.0\tables\qchan_v400_analysis_features.csv',
        ],
        patterns=['qchan_v400_analysis_features.parquet', 'qchan_v400_analysis_features.csv'],
        label='QCHAN analysis features',
    ),
    'QDIST': resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            'nonlinear_distortion/qdist-v4.1.0/finalization/tables/qdist_v410_analysis_features.parquet',
            'nonlinear_distortion/qdist-v4.1.0/finalization/tables/qdist_v410_analysis_features.csv',
            'nonlinear_distortion/qdist-v4.1.0/tables/qdist_v410_analysis_features.parquet',
            'nonlinear_distortion/qdist-v4.1.0/tables/qdist_v410_analysis_features.csv',
        ],
        patterns=['qdist_v410_analysis_features.parquet', 'qdist_v410_analysis_features.csv'],
        label='QDIST finalized analysis features',
    ),
    'QTEMP': resolve_table(
        FAMILY_FREEZE_DIR,
        exact_relatives=[
            r'temporal_discontinuity\qtemp-v1.0.0-analytical-final-no-retained\tables\qtemp_v100_exploratory_features.csv',
        ],
        patterns=['qtemp_v100_exploratory_features.csv'],
        label='QTEMP exploratory features',
    ),
}

input_inventory = pd.DataFrame([
    {
        'input': 'metadata', 'path': str(metadata_path),
        'sha256': sha256_file(metadata_path), 'bytes': metadata_path.stat().st_size,
    },
    *[
        {
            'input': family, 'path': str(path),
            'sha256': sha256_file(path), 'bytes': path.stat().st_size,
        }
        for family, path in family_paths.items()
    ],
])
display(input_inventory)
export_dataframe(input_inventory, MANIFEST_DIR / 'input_inventory.csv')

metadata_raw = read_table(metadata_path)
family_raw = {family: read_table(path) for family, path in family_paths.items()}

print('Metadata:', metadata_raw.shape)
for family, frame in family_raw.items():
    print(f'{family}: {frame.shape}')


## 3. Frozen feature registry, structural dependencies, and PCA sets


In [ ]:
FEATURE_REGISTRY = pd.DataFrame([
    # QADD
    dict(feature='qadd_pause_ac_level_dbfs_median', family='QADD', label='Pause-region level',
         short_label='Pause-region level', unit='dBFS', validated_role='primary_contextual',
         inference_tier='core', transform='none', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qadd_pause_level_support_tier',
         exposure_field=None, direction='higher recorded pause-region level; contextual',
         zero_semantics='digital full scale reference; zero is not a cleanliness threshold',
         include_main=True, include_persistence=True),
    dict(feature='qadd_pause_level_iqr_db', family='QADD', label='Pause-level variability',
         short_label='Pause-level IQR', unit='dB', validated_role='secondary',
         inference_tier='secondary', transform='log1p', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qadd_pause_dispersion_support_tier',
         exposure_field=None, direction='higher pause-level variability',
         zero_semantics='measured zero variability where available',
         include_main=True, include_persistence=True),
    dict(feature='qadd_speech_pause_level_contrast_db', family='QADD', label='Speech-pause level contrast',
         short_label='Speech-pause contrast', unit='dB', validated_role='secondary_mixed',
         inference_tier='secondary', transform='none', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qadd_speech_pause_contrast_support_tier',
         exposure_field=None, direction='higher speech-pause separation; not an adverse-quality direction',
         zero_semantics='equal speech and pause levels in the defined view',
         include_main=True, include_persistence=True),
    dict(feature='qadd_pause_spectral_flatness', family='QADD', label='Pause spectral flatness',
         short_label='Pause spectral flatness', unit='ratio', validated_role='secondary_nonordinal',
         inference_tier='secondary', transform='none', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qadd_flatness_support_tier',
         exposure_field=None, direction='nonordinal spectral structure',
         zero_semantics='structured/tonal extreme; not clean',
         include_main=True, include_persistence=True),
    dict(feature='qadd_mains_hum_comb_score_db', family='QADD', label='Hum-comb prominence',
         short_label='Hum-comb prominence', unit='dB', validated_role='targeted_conditional',
         inference_tier='secondary', transform='asinh', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qadd_hum_support_tier',
         exposure_field=None, direction='higher hum-pattern prominence',
         zero_semantics='no excess on this score; not proof of no hum',
         include_main=True, include_persistence=True),

    # QGAIN
    dict(feature='qgain_typical_speech_level_dbfs', family='QGAIN', label='Typical speech level',
         short_label='Typical speech level', unit='dBFS', validated_role='primary_contextual',
         inference_tier='core', transform='none', outcome_class='continuous',
         availability_field='qgain_typical_speech_level_dbfs_available', censor_field=None,
         support_field='qgain_typical_speech_level_dbfs_support_tier', exposure_field=None,
         direction='higher digital level (less negative dBFS); contextual',
         zero_semantics='digital full scale reference; not calibrated SPL',
         include_main=True, include_persistence=True),
    dict(feature='qgain_within_segment_iqr_db', family='QGAIN', label='Within-segment level IQR',
         short_label='Within-segment IQR', unit='dB', validated_role='primary_mixed',
         inference_tier='core', transform='log1p', outcome_class='continuous',
         availability_field='qgain_within_segment_iqr_db_available', censor_field=None,
         support_field='qgain_within_segment_iqr_db_support_tier', exposure_field=None,
         direction='higher within-segment level variability',
         zero_semantics='no measured within-segment dispersion',
         include_main=True, include_persistence=True),
    dict(feature='qgain_between_segment_mad_db', family='QGAIN', label='Between-segment level MAD',
         short_label='Between-segment MAD', unit='dB', validated_role='secondary_mixed',
         inference_tier='secondary', transform='log1p', outcome_class='continuous',
         availability_field='qgain_between_segment_mad_db_available', censor_field=None,
         support_field='qgain_between_segment_mad_db_support_tier', exposure_field=None,
         direction='higher between-segment level variability',
         zero_semantics='no measured between-segment dispersion',
         include_main=True, include_persistence=True),
    dict(feature='qgain_abs_drift_db_per_min', family='QGAIN', label='Absolute level drift',
         short_label='Absolute level drift', unit='dB/min', validated_role='exploratory_contextual',
         inference_tier='exploratory', transform='log1p', outcome_class='continuous',
         availability_field='qgain_abs_drift_db_per_min_available', censor_field=None,
         support_field='qgain_abs_drift_db_per_min_support_tier', exposure_field=None,
         direction='higher absolute monotonic level drift',
         zero_semantics='no estimated monotonic drift',
         include_main=False, include_persistence=True),

    # QREV
    dict(feature='qrev_tail_excess_100ms_db', family='QREV', label='Early residual-tail excess',
         short_label='Early tail excess†', unit='dB', validated_role='primary_conditional',
         inference_tier='core', transform='asinh', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qrev_tail_excess_100ms_db_support_tier',
         exposure_field=None, direction='higher early residual energy',
         zero_semantics='early and late regions equal on the defined scale',
         include_main=True, include_persistence=True),
    dict(feature='qrev_tail_persistence_median_sec', family='QREV', label='Tail persistence',
         short_label='Tail persistence†', unit='s', validated_role='secondary_conditional_nonindependent',
         inference_tier='secondary', transform='log1p', outcome_class='censored_continuous',
         availability_field=None, censor_field='qrev_persistence_recording_median_censored',
         support_field='qrev_tail_persistence_median_sec_support_tier', exposure_field=None,
         direction='higher residual persistence',
         zero_semantics='immediate return within estimator resolution',
         include_main=True, include_persistence=True),
    dict(feature='qrev_downward_decay_rate_db_per_sec', family='QREV', label='Downward residual decay rate',
         short_label='Downward decay rate†', unit='dB/s', validated_role='exploratory_conditional',
         inference_tier='exploratory', transform='log1p', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qrev_downward_decay_rate_db_per_sec_support_tier',
         exposure_field=None, direction='higher means faster downward decay, not more reverberation',
         zero_semantics='conditional on a valid decline',
         include_main=False, include_persistence=True),
    dict(feature='qrev_srmr_norm', family='QREV', label='Normalized-fast SRMR',
         short_label='Normalized-fast SRMR', unit='ratio', validated_role='established_comparator',
         inference_tier='core', transform='log1p', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qrev_srmr_norm_support_tier',
         exposure_field=None, direction='higher generally indicates less modulation smearing',
         zero_semantics='theoretical lower bound; not a clean/unclean state',
         include_main=True, include_persistence=True),

    # QCHAN
    dict(feature='qchan_ltas_distance_db', family='QCHAN', label='LTAS distance',
         short_label='LTAS distance', unit='dB RMS', validated_role='primary_nonordinal',
         inference_tier='core', transform='log1p', outcome_class='continuous',
         availability_field=None, censor_field=None, support_field='qchan_ltas_distance_db_support_tier',
         exposure_field=None, direction='higher reference-relative spectral deviation; nonordinal',
         zero_semantics='exact match to the reference spectral shape',
         include_main=True, include_persistence=True),
    dict(feature='qchan_rolloff95_deficit_hz', family='QCHAN', label='Rolloff-95 deficit',
         short_label='Rolloff-95 deficit', unit='Hz', validated_role='primary_one_sided',
         inference_tier='core', transform='log1p', outcome_class='one_sided_zero',
         availability_field=None, censor_field=None, support_field='qchan_rolloff95_deficit_hz_support_tier',
         exposure_field=None, direction='higher upper-frequency deficit',
         zero_semantics='target rolloff equal to or above the reference',
         include_main=True, include_persistence=True),
    dict(feature='qchan_highband_ratio_deficit', family='QCHAN', label='High-band ratio deficit',
         short_label='High-band ratio deficit', unit='proportion', validated_role='secondary_nonindependent',
         inference_tier='secondary', transform='log1p', outcome_class='one_sided_zero',
         availability_field=None, censor_field=None, support_field='qchan_highband_ratio_deficit_support_tier',
         exposure_field=None, direction='higher high-band deficit',
         zero_semantics='target high-band share equal to or above the reference',
         include_main=True, include_persistence=True),
    dict(feature='qchan_tilt_steepening_db_per_oct', family='QCHAN', label='Spectral-tilt steepening',
         short_label='Spectral-tilt steepening', unit='dB/oct', validated_role='exploratory_phenotype_sensitive',
         inference_tier='exploratory', transform='log1p', outcome_class='one_sided_zero',
         availability_field=None, censor_field=None, support_field='qchan_tilt_steepening_db_per_oct_support_tier',
         exposure_field=None, direction='higher reference-relative tilt steepening',
         zero_semantics='target tilt equal to or shallower than the reference',
         include_main=False, include_persistence=True),

    # QDIST
    dict(feature='qdist_hard_clipped_sample_fraction', family='QDIST', label='Hard-clipped sample fraction',
         short_label='Hard-clipping events', unit='fraction', validated_role='primary_sparse_burden',
         inference_tier='core', transform='log1p', outcome_class='sparse_event_extent',
         availability_field='qdist_available', censor_field=None, support_field='qdist_support_tier',
         exposure_field='qdist_finite_exposure_sec', direction='higher detected hard-plateau burden',
         zero_semantics='no detector-defined hard-plateau support observed',
         include_main=True, include_persistence=False),
    dict(feature='qdist_hard_clip_event_rate_per_min', family='QDIST', label='Hard-clipping episode rate',
         short_label='Hard-clipping episode rate', unit='episodes/min', validated_role='secondary_sparse_event',
         inference_tier='secondary', transform='log1p', outcome_class='sparse_event_rate',
         availability_field='qdist_available', censor_field=None, support_field='qdist_support_tier',
         exposure_field='qdist_finite_exposure_sec', direction='higher detected episode frequency',
         zero_semantics='no detector-defined episode observed',
         include_main=False, include_persistence=False),
    dict(feature='qdist_hard_clipped_frame_fraction', family='QDIST', label='Hard-clipped frame fraction',
         short_label='Hard-clipped frame fraction', unit='fraction', validated_role='conditional_audit',
         inference_tier='audit', transform='log1p', outcome_class='audit',
         availability_field='qdist_available', censor_field=None, support_field='qdist_support_tier',
         exposure_field='qdist_finite_exposure_sec', direction='audit only',
         zero_semantics='no grid frame intersected an accepted plateau',
         include_main=False, include_persistence=False),

    # QTEMP
    dict(feature='qtemp_dropout_duration_fraction', family='QTEMP', label='Dropout-like duration fraction',
         short_label='Dropout-like events', unit='fraction', validated_role='exploratory_not_primary',
         inference_tier='exploratory', transform='log1p', outcome_class='monitoring',
         availability_field=None, censor_field=None, support_field=None, exposure_field=None,
         direction='exploratory detected burden', zero_semantics='no event met the registered dropout-like rule',
         include_main=True, include_persistence=False),
    dict(feature='qtemp_dropout_event_rate_per_min', family='QTEMP', label='Dropout-like event rate',
         short_label='Dropout-like event rate', unit='events/min', validated_role='exploratory_not_primary',
         inference_tier='exploratory', transform='log1p', outcome_class='monitoring',
         availability_field=None, censor_field=None, support_field=None, exposure_field=None,
         direction='exploratory detected frequency', zero_semantics='no event met the registered dropout-like rule',
         include_main=False, include_persistence=False),
    dict(feature='qtemp_frozen_audio_duration_fraction', family='QTEMP', label='Frozen-audio duration fraction',
         short_label='Frozen-audio events', unit='fraction', validated_role='monitoring_zero_variation',
         inference_tier='monitoring', transform='log1p', outcome_class='monitoring',
         availability_field=None, censor_field=None, support_field=None, exposure_field=None,
         direction='monitoring only', zero_semantics='no event met the registered repetition rule',
         include_main=True, include_persistence=False),
    dict(feature='qtemp_frozen_audio_event_rate_per_min', family='QTEMP', label='Frozen-audio event rate',
         short_label='Frozen-audio event rate', unit='events/min', validated_role='monitoring_zero_variation',
         inference_tier='monitoring', transform='log1p', outcome_class='monitoring',
         availability_field=None, censor_field=None, support_field=None, exposure_field=None,
         direction='monitoring only', zero_semantics='no event met the registered repetition rule',
         include_main=False, include_persistence=False),
])
FEATURE_REGISTRY['feature_order'] = np.arange(len(FEATURE_REGISTRY))
FEATURE_REGISTRY['is_core'] = FEATURE_REGISTRY['inference_tier'].eq('core')

required_registry_fields = [
    'feature', 'family', 'label', 'short_label', 'unit', 'validated_role',
    'inference_tier', 'transform', 'outcome_class', 'availability_field',
    'censor_field', 'support_field', 'exposure_field', 'direction',
    'zero_semantics', 'include_main', 'include_persistence', 'feature_order',
]
assert not FEATURE_REGISTRY[required_registry_fields].isna().all(axis=1).any()
display(FEATURE_REGISTRY)
export_dataframe(FEATURE_REGISTRY, TABLE_DIR / 'feature_registry.csv')


FEATURE_CODES = {
    'qadd_pause_ac_level_dbfs_median': 'A1',
    'qadd_pause_level_iqr_db': 'A2',
    'qadd_speech_pause_level_contrast_db': 'A3',
    'qadd_pause_spectral_flatness': 'A4',
    'qadd_mains_hum_comb_score_db': 'A5',
    'qgain_typical_speech_level_dbfs': 'G1',
    'qgain_within_segment_iqr_db': 'G2',
    'qgain_between_segment_mad_db': 'G3',
    'qgain_abs_drift_db_per_min': 'G4',
    'qrev_tail_excess_100ms_db': 'R1',
    'qrev_tail_persistence_median_sec': 'R2',
    'qrev_downward_decay_rate_db_per_sec': 'R3',
    'qrev_srmr_norm': 'R4',
    'qchan_ltas_distance_db': 'C1',
    'qchan_rolloff95_deficit_hz': 'C2',
    'qchan_highband_ratio_deficit': 'C3',
    'qchan_tilt_steepening_db_per_oct': 'C4',
    'qdist_hard_clipped_sample_fraction': 'D1',
    'qdist_hard_clip_event_rate_per_min': 'D2',
    'qtemp_dropout_duration_fraction': 'T1',
    'qtemp_dropout_event_rate_per_min': 'T2',
    'qtemp_frozen_audio_duration_fraction': 'T3',
    'qtemp_frozen_audio_event_rate_per_min': 'T4',
}
FEATURE_REGISTRY['feature_code'] = FEATURE_REGISTRY['feature'].map(FEATURE_CODES)

# Explicit same-ledger relationships are governed before correlations are inspected.
STRUCTURAL_DEPENDENCY_PAIRS = {
    frozenset({'qadd_pause_ac_level_dbfs_median', 'qadd_speech_pause_level_contrast_db'}),
    frozenset({'qrev_tail_excess_100ms_db', 'qrev_tail_persistence_median_sec'}),
    frozenset({'qrev_tail_excess_100ms_db', 'qrev_downward_decay_rate_db_per_sec'}),
    frozenset({'qrev_tail_persistence_median_sec', 'qrev_downward_decay_rate_db_per_sec'}),
    frozenset({'qdist_hard_clipped_sample_fraction', 'qdist_hard_clip_event_rate_per_min'}),
    frozenset({'qtemp_dropout_duration_fraction', 'qtemp_dropout_event_rate_per_min'}),
    frozenset({'qtemp_frozen_audio_duration_fraction', 'qtemp_frozen_audio_event_rate_per_min'}),
}

PRIMARY_PCA_FEATURES = [
    'qadd_pause_ac_level_dbfs_median',
    'qadd_pause_level_iqr_db',
    'qgain_typical_speech_level_dbfs',
    'qgain_within_segment_iqr_db',
    'qgain_between_segment_mad_db',
    'qrev_srmr_norm',
    'qchan_ltas_distance_db',
    'qchan_rolloff95_deficit_hz',
]

PCA_SENSITIVITY_FEATURES = {
    'add_highband_deficit': PRIMARY_PCA_FEATURES + ['qchan_highband_ratio_deficit'],
    'replace_rolloff_with_highband': [
        feature for feature in PRIMARY_PCA_FEATURES
        if feature != 'qchan_rolloff95_deficit_hz'
    ] + ['qchan_highband_ratio_deficit'],
}

export_dataframe(FEATURE_REGISTRY, TABLE_DIR / 'feature_registry.csv')


## 4. Harmonize family tables and construct the common analysis matrix


In [ ]:
def select_family_columns(frame: pd.DataFrame, family_prefix: str) -> pd.DataFrame:
    if 'logical_recording_id' not in frame.columns:
        raise KeyError(f'{family_prefix}: logical_recording_id is missing.')
    columns = ['logical_recording_id'] + [
        column for column in frame.columns
        if column.lower().startswith(family_prefix.lower())
    ]
    selected = frame.loc[:, list(dict.fromkeys(columns))].copy()
    selected['logical_recording_id'] = normalize_id(selected['logical_recording_id'])
    if selected['logical_recording_id'].duplicated().any():
        examples = selected.loc[
            selected['logical_recording_id'].duplicated(keep=False),
            'logical_recording_id',
        ].head(10).tolist()
        raise AssertionError(f'{family_prefix}: duplicated recording IDs: {examples}')
    return selected

family_tables = {
    family: select_family_columns(frame, family.lower())
    for family, frame in family_raw.items()
}

id_sets = {
    family: set(frame['logical_recording_id'].dropna())
    for family, frame in family_tables.items()
}
intersection_ids = set.intersection(*id_sets.values())
union_ids = set.union(*id_sets.values())

family_id_summary = pd.DataFrame([
    {
        'family': family,
        'rows': len(frame),
        'unique_recordings': frame['logical_recording_id'].nunique(),
        'missing_from_union': len(union_ids - ids),
    }
    for (family, frame), ids in zip(family_tables.items(), id_sets.values())
])
display(family_id_summary)

if len(intersection_ids) != 519 or len(union_ids) != 519:
    discrepancy = []
    for recording_id in sorted(union_ids):
        missing = [family for family, ids in id_sets.items() if recording_id not in ids]
        if missing:
            discrepancy.append({
                'logical_recording_id': recording_id,
                'missing_families': '|'.join(missing),
            })
    discrepancy_frame = pd.DataFrame(discrepancy)
    export_dataframe(discrepancy_frame, LOG_DIR / 'family_recording_id_discrepancies.csv')
    raise AssertionError('The six family tables do not define one common 519-recording population.')

feature_matrix = pd.DataFrame({'logical_recording_id': sorted(intersection_ids)})
for family in ['QADD', 'QGAIN', 'QREV', 'QCHAN', 'QDIST', 'QTEMP']:
    feature_matrix = feature_matrix.merge(
        family_tables[family], on='logical_recording_id', how='left', validate='one_to_one'
    )

missing_features = [
    feature for feature in FEATURE_REGISTRY['feature']
    if feature not in feature_matrix.columns
]
if missing_features:
    raise KeyError(f'Required feature columns are missing: {missing_features}')

metadata = metadata_raw.copy()
if 'logical_recording_id' not in metadata.columns:
    raise KeyError('Metadata ledger does not contain logical_recording_id.')
metadata['logical_recording_id'] = normalize_id(metadata['logical_recording_id'])
if metadata['logical_recording_id'].duplicated().any():
    raise AssertionError('Metadata contains duplicated logical_recording_id values.')
metadata = metadata.loc[metadata['logical_recording_id'].isin(intersection_ids)].copy()

participant_source = 'SubjectID' if 'SubjectID' in metadata.columns else 'subject'
metadata['participant_id'] = normalize_id(metadata[participant_source])
metadata['cohort'] = (
    metadata['diagnosis_analysis'].astype('string').str.upper()
    .replace({'CONTROLS': 'CONTROL', 'CONTROL': 'CONTROL', 'ALS': 'ALS'})
)
metadata['cohort_als'] = metadata['cohort'].map({'CONTROL': 0.0, 'ALS': 1.0})
metadata['recording_date'] = pd.to_datetime(
    metadata.get('recording_date_analysis', metadata.get('Recording date')),
    errors='coerce',
)
metadata['age_years'] = safe_numeric(metadata.get('age_at_recording_years'))
metadata['sex'] = metadata.get('Sex', pd.Series(index=metadata.index, dtype='object')).astype('string').str.upper()
metadata['sample_rate_hz'] = safe_numeric(metadata.get('Sampling Rate'))
metadata['protocol_id'] = metadata.get('Protocol ID', pd.Series(index=metadata.index, dtype='object')).astype('string')
metadata['bulbar_score'] = safe_numeric(metadata.get('ALSFRS bulbar subscore'))
metadata['assessment_recording_delta_days'] = safe_numeric(metadata.get('assessment_recording_delta_days'))

if 'assessment_within_primary_window' in metadata.columns:
    window_flag = boolean_series(metadata['assessment_within_primary_window'])
    metadata['bulbar_score_matched'] = metadata['bulbar_score'].where(window_flag)
else:
    metadata['bulbar_score_matched'] = metadata['bulbar_score']

analysis = metadata.merge(
    feature_matrix, on='logical_recording_id', how='inner', validate='one_to_one'
)
analysis = analysis.sort_values(
    ['participant_id', 'recording_date', 'logical_recording_id'], kind='stable'
).reset_index(drop=True)
analysis['recording_sequence'] = analysis.groupby('participant_id').cumcount() + 1
first_date = analysis.groupby('participant_id')['recording_date'].transform('min')
analysis['time_since_first_years'] = (
    (analysis['recording_date'] - first_date).dt.total_seconds()
    / (365.25 * 24 * 60 * 60)
)

print('Analysis matrix:', analysis.shape)


## 5. Blocking population and metadata audit


In [ ]:
audit_rows = []

def add_audit(check: str, observed: Any, expected: Any, passed: bool, detail: str = '', blocking: bool = True):
    audit_rows.append({
        'check': check, 'observed': observed, 'expected': expected,
        'passed': bool(passed), 'blocking': bool(blocking), 'detail': detail,
    })

add_audit('recording count', len(analysis), 519, len(analysis) == 519)
add_audit('participant count', analysis['participant_id'].nunique(), 224,
          analysis['participant_id'].nunique() == 224)
add_audit('unique recording IDs', analysis['logical_recording_id'].nunique(), 519,
          analysis['logical_recording_id'].nunique() == 519)
add_audit('one participant per recording',
          analysis.groupby('logical_recording_id')['participant_id'].nunique().max(), 1,
          analysis.groupby('logical_recording_id')['participant_id'].nunique().max() == 1)
add_audit('cohort constant within participant',
          analysis.groupby('participant_id')['cohort'].nunique().max(), 1,
          analysis.groupby('participant_id')['cohort'].nunique().max() == 1)
add_audit('cohort values', sorted(analysis['cohort'].dropna().unique().tolist()),
          ['ALS', 'CONTROL'], set(analysis['cohort'].dropna()) == {'ALS', 'CONTROL'})
add_audit('missing participant ID', int(analysis['participant_id'].isna().sum()), 0,
          analysis['participant_id'].isna().sum() == 0)
add_audit('missing cohort', int(analysis['cohort'].isna().sum()), 0,
          analysis['cohort'].isna().sum() == 0)
add_audit(
    'missing recording date', int(analysis['recording_date'].isna().sum()),
    'reported; excluded from time-dependent analyses', True,
    detail='Missing dates do not block descriptive/cohort analyses.', blocking=False,
)
if 'freeze_included' in analysis.columns:
    add_audit('all metadata rows freeze included',
              int(analysis['freeze_included'].fillna(False).sum()), 519,
              analysis['freeze_included'].fillna(False).all())

population_audit = pd.DataFrame(audit_rows)
display(population_audit)
export_dataframe(population_audit, LOG_DIR / 'population_audit.csv')
blocking_failures = population_audit.loc[population_audit['blocking'] & ~population_audit['passed']]
if not blocking_failures.empty:
    raise AssertionError('Blocking population audit failed. Inferential analysis is prohibited.')

cohort_summary = (
    analysis.groupby('cohort', observed=True)
    .agg(recordings=('logical_recording_id', 'nunique'),
         participants=('participant_id', 'nunique'))
    .reset_index()
)
recording_count_summary = (
    analysis.groupby(['participant_id', 'cohort'], observed=True)
    .size().rename('recordings').reset_index()
    .groupby('cohort', observed=True)['recordings']
    .agg(['count', 'median', 'min', 'max']).reset_index()
)

als = analysis.loc[analysis['cohort'] == 'ALS'].copy()
bulbar_counts = als.groupby('participant_id')['bulbar_score_matched'].count()
bulbar_distinct = als.groupby('participant_id')['bulbar_score_matched'].nunique()
bulbar_audit = pd.DataFrame({
    'quantity': [
        'ALS recordings', 'ALS recordings with matched bulbar score',
        'ALS participants', 'ALS participants with matched bulbar score',
        'ALS participants with >=2 matched scores',
        'ALS participants with >=2 distinct matched scores',
        'Median absolute assessment-recording difference (days)',
    ],
    'value': [
        len(als), als['bulbar_score_matched'].notna().sum(),
        als['participant_id'].nunique(),
        als.loc[als['bulbar_score_matched'].notna(), 'participant_id'].nunique(),
        bulbar_counts.ge(2).sum(), bulbar_distinct.ge(2).sum(),
        als.loc[als['bulbar_score_matched'].notna(), 'assessment_recording_delta_days'].abs().median(),
    ],
})

display(cohort_summary)
display(recording_count_summary)
display(bulbar_audit)
export_dataframe(cohort_summary, TABLE_DIR / 'population_by_cohort.csv')
export_dataframe(recording_count_summary, TABLE_DIR / 'recordings_per_participant.csv')
export_dataframe(bulbar_audit, TABLE_DIR / 'bulbar_linkage_audit.csv')


## 6. Availability, frozen transformations, scaling, censoring, and zero-state construction


In [ ]:
scaling_rows = []

for row in FEATURE_REGISTRY.itertuples(index=False):
    feature = row.feature
    available_col = f'{feature}__available'
    transformed_col = f'{feature}__transformed'
    z_col = f'{feature}__z'
    raw_z_col = f'{feature}__raw_z'
    positive_col = f'{feature}__positive'

    value_available = analysis[feature].notna()
    if row.availability_field and row.availability_field in analysis.columns:
        explicit_available = boolean_series(analysis[row.availability_field])
        analysis[available_col] = explicit_available & value_available
    else:
        analysis[available_col] = value_available

    transformed = transform_feature(analysis[feature], row.transform).where(analysis[available_col])
    analysis[transformed_col] = transformed
    z, transformed_median, transformed_iqr = robust_scale(transformed)
    analysis[z_col] = z

    raw_values = safe_numeric(analysis[feature]).where(analysis[available_col])
    raw_z, raw_median, raw_iqr = robust_scale(raw_values)
    analysis[raw_z_col] = raw_z
    analysis[positive_col] = analysis[available_col] & safe_numeric(analysis[feature]).gt(0)

    if row.censor_field and row.censor_field in analysis.columns:
        analysis[f'{feature}__censored'] = boolean_series(analysis[row.censor_field]) & analysis[available_col]
    else:
        analysis[f'{feature}__censored'] = False

    scaling_rows.append({
        'feature': feature,
        'family': row.family,
        'transform': row.transform,
        'n_available': int(analysis[available_col].sum()),
        'transformed_median': transformed_median,
        'transformed_iqr': transformed_iqr,
        'raw_median': raw_median,
        'raw_iqr': raw_iqr,
        'degenerate_transformed': not (np.isfinite(transformed_iqr) and transformed_iqr > 0),
        'degenerate_raw': not (np.isfinite(raw_iqr) and raw_iqr > 0),
        'n_censored': int(analysis[f'{feature}__censored'].sum()),
    })

scaling_registry = pd.DataFrame(scaling_rows)
display(scaling_registry)
export_dataframe(scaling_registry, TABLE_DIR / 'scaling_registry.csv')

blocking_degenerate = scaling_registry.merge(
    FEATURE_REGISTRY[['feature', 'outcome_class', 'inference_tier']], on='feature', how='left'
).loc[
    lambda frame: frame['outcome_class'].isin(['continuous', 'censored_continuous', 'one_sided_zero'])
    & frame['inference_tier'].isin(['core', 'secondary'])
    & frame['degenerate_transformed']
]
if not blocking_degenerate.empty:
    raise AssertionError(
        'A core/supporting continuous feature is degenerate after transformation: '
        f"{blocking_degenerate['feature'].tolist()}"
    )


## 7. Structural feature eligibility and pairwise support audit

The full pairwise set is determined without inspecting correlations. A feature must be retained, nonconstant after its frozen transformation, and sufficiently supported. Sparse QDIST and QTEMP outputs require at least 20 positive recordings from at least 10 participants; otherwise they remain descriptive and are excluded. Pairwise support is then evaluated separately for between- and within-participant estimands.


In [ ]:
STRUCTURE_ALLOWED_OUTCOMES = {'continuous', 'censored_continuous', 'one_sided_zero'}

eligibility_rows = []
for row in FEATURE_REGISTRY.itertuples(index=False):
    feature = row.feature
    z_col = f'{feature}__z'
    available_col = f'{feature}__available'
    positive_col = f'{feature}__positive'
    n_available = int(analysis[available_col].sum())
    n_participants = int(analysis.loc[analysis[available_col], 'participant_id'].nunique())
    n_unique = int(analysis[z_col].nunique(dropna=True))
    n_positive = int(analysis[positive_col].sum())
    n_positive_participants = int(
        analysis.loc[analysis[positive_col], 'participant_id'].nunique()
    )

    base_allowed = (
        row.outcome_class in STRUCTURE_ALLOWED_OUTCOMES
        and row.inference_tier not in {'audit', 'monitoring'}
        and n_unique >= 2
        and n_participants >= MIN_BETWEEN_PARTICIPANTS
    )
    sparse_support = True
    if row.outcome_class in {'sparse_event_extent', 'sparse_event_rate', 'monitoring'}:
        sparse_support = (
            n_positive >= MIN_SPARSE_POSITIVE_RECORDINGS
            and n_positive_participants >= MIN_SPARSE_POSITIVE_PARTICIPANTS
        )

    include_pairwise = bool(base_allowed and sparse_support)
    exclusion_reason = ''
    if not base_allowed:
        if row.outcome_class not in STRUCTURE_ALLOWED_OUTCOMES:
            exclusion_reason = f'Outcome class {row.outcome_class} is not eligible.'
        elif row.inference_tier in {'audit', 'monitoring'}:
            exclusion_reason = f'Role {row.inference_tier} is not inferential.'
        elif n_unique < 2:
            exclusion_reason = 'Degenerate after frozen transformation.'
        elif n_participants < MIN_BETWEEN_PARTICIPANTS:
            exclusion_reason = 'Insufficient participant support.'
    elif not sparse_support:
        exclusion_reason = 'Sparse positive support below the prespecified event gate.'

    eligibility_rows.append({
        'feature': feature,
        'feature_code': row.feature_code,
        'family': row.family,
        'label': row.short_label,
        'validated_role': row.validated_role,
        'inference_tier': row.inference_tier,
        'outcome_class': row.outcome_class,
        'n_available_recordings': n_available,
        'n_available_participants': n_participants,
        'n_unique_transformed_values': n_unique,
        'n_positive_recordings': n_positive,
        'n_positive_participants': n_positive_participants,
        'include_pairwise': include_pairwise,
        'exclusion_reason': exclusion_reason,
    })

structural_eligibility = pd.DataFrame(eligibility_rows)
PAIRWISE_FEATURES = structural_eligibility.loc[
    structural_eligibility['include_pairwise'], 'feature'
].tolist()

if len(PAIRWISE_FEATURES) != 17:
    warnings.warn(
        f'Expected 17 QADD-QCHAN pairwise features; found {len(PAIRWISE_FEATURES)}. '
        'Review the eligibility audit before interpreting results.',
        RuntimeWarning,
    )


def pair_key(feature_x: str, feature_y: str) -> frozenset[str]:
    return frozenset({feature_x, feature_y})


def prepare_between_profiles(
    data: pd.DataFrame,
    feature_x: str,
    feature_y: str,
    *,
    summary: str = 'mean',
) -> pd.DataFrame:
    columns = [
        'participant_id',
        f'{feature_x}__transformed',
        f'{feature_y}__transformed',
    ]
    joint = data[columns].dropna().copy()
    aggregation = 'mean' if summary == 'mean' else 'median'
    profiles = joint.groupby('participant_id', observed=True)[columns[1:]].agg(aggregation)
    return profiles.dropna()


def prepare_within_pair(
    data: pd.DataFrame,
    feature_x: str,
    feature_y: str,
) -> pd.DataFrame:
    x_col = f'{feature_x}__transformed'
    y_col = f'{feature_y}__transformed'
    joint = data[['participant_id', x_col, y_col]].dropna().copy()
    counts = joint.groupby('participant_id', observed=True).size()
    joint = joint.loc[joint['participant_id'].isin(counts[counts >= 2].index)].copy()
    if joint.empty:
        return joint
    variation = joint.groupby('participant_id', observed=True)[[x_col, y_col]].nunique()
    valid_participants = variation.index[
        variation[x_col].ge(2) & variation[y_col].ge(2)
    ]
    return joint.loc[joint['participant_id'].isin(valid_participants)].copy()


pair_support_rows = []
for index, feature_x in enumerate(PAIRWISE_FEATURES):
    for feature_y in PAIRWISE_FEATURES[index + 1:]:
        between = prepare_between_profiles(analysis, feature_x, feature_y)
        within = prepare_within_pair(analysis, feature_x, feature_y)
        n_between = len(between)
        n_within_recordings = len(within)
        n_within_participants = within['participant_id'].nunique() if not within.empty else 0
        pair_support_rows.append({
            'feature_x': feature_x,
            'feature_y': feature_y,
            'code_x': FEATURE_CODES[feature_x],
            'code_y': FEATURE_CODES[feature_y],
            'family_x': FEATURE_REGISTRY.set_index('feature').loc[feature_x, 'family'],
            'family_y': FEATURE_REGISTRY.set_index('feature').loc[feature_y, 'family'],
            'same_family': (
                FEATURE_REGISTRY.set_index('feature').loc[feature_x, 'family']
                == FEATURE_REGISTRY.set_index('feature').loc[feature_y, 'family']
            ),
            'structural_dependency': pair_key(feature_x, feature_y) in STRUCTURAL_DEPENDENCY_PAIRS,
            'n_between_participants': n_between,
            'between_gate': n_between >= MIN_BETWEEN_PARTICIPANTS,
            'n_within_joint_recordings': n_within_recordings,
            'n_within_repeated_participants': n_within_participants,
            'within_gate': (
                n_within_recordings >= MIN_WITHIN_JOINT_RECORDINGS
                and n_within_participants >= MIN_WITHIN_REPEATED_PARTICIPANTS
            ),
        })

pair_support = pd.DataFrame(pair_support_rows)

# Blocking PCA support audit.
primary_pca_z = [f'{feature}__z' for feature in PRIMARY_PCA_FEATURES]
pca_recording_complete = analysis.dropna(subset=primary_pca_z).copy()
pca_participant_profiles = (
    analysis.groupby('participant_id', observed=True)[primary_pca_z]
    .mean().dropna()
)
pca_repeated_counts = pca_recording_complete.groupby('participant_id', observed=True).size()

pca_support_audit = pd.DataFrame([
    {'check': 'primary PCA features present', 'observed': len(PRIMARY_PCA_FEATURES), 'expected': 8,
     'passed': len(PRIMARY_PCA_FEATURES) == 8},
    {'check': 'between-PCA complete participants', 'observed': len(pca_participant_profiles),
     'expected': f'>={PCA_MIN_COMPLETE_PARTICIPANTS}',
     'passed': len(pca_participant_profiles) >= PCA_MIN_COMPLETE_PARTICIPANTS},
    {'check': 'within-PCA repeated participants', 'observed': int(pca_repeated_counts.ge(2).sum()),
     'expected': f'>={PCA_MIN_REPEATED_PARTICIPANTS}',
     'passed': int(pca_repeated_counts.ge(2).sum()) >= PCA_MIN_REPEATED_PARTICIPANTS},
    {'check': 'within-PCA complete recordings', 'observed': len(pca_recording_complete),
     'expected': 'reported', 'passed': len(pca_recording_complete) > 0},
])

for frame, name in [
    (structural_eligibility, 'structural_feature_eligibility.csv'),
    (pair_support, 'pairwise_support_audit.csv'),
    (pca_support_audit, 'pca_support_audit.csv'),
]:
    export_dataframe(frame, TABLE_DIR / name)

display(structural_eligibility)
display(pair_support.head(20))
display(pca_support_audit)

if not pca_support_audit['passed'].all():
    raise AssertionError('Primary PCA support audit failed. PCA execution is prohibited.')


## 8. Correlation estimators and participant-cluster bootstrap

Between-participant correlations use participant means of frozen transformed values. Within-participant correlations rank the jointly available recording values, remove participant-specific mean ranks, and calculate a weighted correlation in which every participant contributes equal total weight. Participant bootstrap samples are treated as new cluster copies when a participant is selected more than once.


In [ ]:
def weighted_correlation(x: np.ndarray, y: np.ndarray, weights: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(weights) & (weights > 0)
    x, y, weights = x[valid], y[valid], weights[valid]
    if len(x) < 3:
        return np.nan
    weights = weights / weights.sum()
    mean_x = np.sum(weights * x)
    mean_y = np.sum(weights * y)
    covariance = np.sum(weights * (x - mean_x) * (y - mean_y))
    variance_x = np.sum(weights * (x - mean_x) ** 2)
    variance_y = np.sum(weights * (y - mean_y) ** 2)
    if variance_x <= 0 or variance_y <= 0:
        return np.nan
    return float(covariance / np.sqrt(variance_x * variance_y))


def weighted_midrank(values: np.ndarray, frequencies: np.ndarray) -> np.ndarray:
    """Midranks equivalent to explicitly repeating each value `frequency` times."""
    values = np.asarray(values, dtype=float)
    frequencies = np.asarray(frequencies, dtype=float)
    ranks = np.full(len(values), np.nan, dtype=float)
    valid = np.isfinite(values) & np.isfinite(frequencies) & (frequencies > 0)
    if valid.sum() == 0:
        return ranks
    valid_indices = np.flatnonzero(valid)
    order = np.argsort(values[valid], kind='mergesort')
    sorted_indices = valid_indices[order]
    sorted_values = values[sorted_indices]
    sorted_frequencies = frequencies[sorted_indices]
    cumulative_before = 0.0
    start = 0
    while start < len(sorted_indices):
        end = start + 1
        while end < len(sorted_indices) and sorted_values[end] == sorted_values[start]:
            end += 1
        group_weight = float(sorted_frequencies[start:end].sum())
        group_rank = cumulative_before + (group_weight + 1.0) / 2.0
        ranks[sorted_indices[start:end]] = group_rank
        cumulative_before += group_weight
        start = end
    return ranks


def weighted_spearman_with_frequencies(
    x: np.ndarray,
    y: np.ndarray,
    frequencies: np.ndarray,
) -> float:
    frequencies = np.asarray(frequencies, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(frequencies) & (frequencies > 0)
    if valid.sum() < 3:
        return np.nan
    x = np.asarray(x, dtype=float)[valid]
    y = np.asarray(y, dtype=float)[valid]
    frequencies = frequencies[valid]
    rank_x = weighted_midrank(x, frequencies)
    rank_y = weighted_midrank(y, frequencies)
    return weighted_correlation(rank_x, rank_y, frequencies)


def within_rank_correlation_arrays(
    x: np.ndarray,
    y: np.ndarray,
    participant_codes: np.ndarray,
    participant_multiplicity: np.ndarray,
    cluster_sizes: np.ndarray,
) -> float:
    """Rank-based within-participant correlation under participant resampling.

    `participant_multiplicity` contains bootstrap selection counts for every global
    participant. The estimator is algebraically equivalent to duplicating all rows
    for every selected participant copy, but avoids expensive DataFrame rebuilding.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    codes = np.asarray(participant_codes, dtype=int)
    frequencies = np.asarray(participant_multiplicity, dtype=float)[codes]
    valid = np.isfinite(x) & np.isfinite(y) & (frequencies > 0)
    if valid.sum() < 3:
        return np.nan
    x, y, codes, frequencies = x[valid], y[valid], codes[valid], frequencies[valid]
    rank_x = weighted_midrank(x, frequencies)
    rank_y = weighted_midrank(y, frequencies)

    weighted_cluster_counts = np.bincount(
        codes, weights=frequencies, minlength=len(participant_multiplicity)
    )
    mean_rank_x = np.divide(
        np.bincount(codes, weights=frequencies * rank_x, minlength=len(participant_multiplicity)),
        weighted_cluster_counts,
        out=np.zeros(len(participant_multiplicity), dtype=float),
        where=weighted_cluster_counts > 0,
    )
    mean_rank_y = np.divide(
        np.bincount(codes, weights=frequencies * rank_y, minlength=len(participant_multiplicity)),
        weighted_cluster_counts,
        out=np.zeros(len(participant_multiplicity), dtype=float),
        where=weighted_cluster_counts > 0,
    )
    centered_x = rank_x - mean_rank_x[codes]
    centered_y = rank_y - mean_rank_y[codes]

    # Each selected participant copy contributes unit total weight. Repeated copies
    # therefore contribute multiplicity / original cluster size at each observation.
    observation_weights = frequencies / cluster_sizes[codes]
    return weighted_correlation(centered_x, centered_y, observation_weights)


def between_spearman(profiles: pd.DataFrame) -> float:
    if len(profiles) < 3 or profiles.iloc[:, 0].nunique() < 2 or profiles.iloc[:, 1].nunique() < 2:
        return np.nan
    return float(stats.spearmanr(profiles.iloc[:, 0], profiles.iloc[:, 1]).statistic)


def within_rank_correlation(
    joint: pd.DataFrame,
    *,
    participant_col: str = 'participant_id',
) -> float:
    if joint.empty:
        return np.nan
    value_columns = [column for column in joint.columns if column != participant_col]
    if len(value_columns) != 2:
        raise ValueError('Within-pair frame must contain one participant column and two values.')
    participant_ids = joint[participant_col].astype('string').to_numpy()
    unique_ids, codes = np.unique(participant_ids, return_inverse=True)
    cluster_sizes_local = np.bincount(codes, minlength=len(unique_ids)).astype(float)
    multiplicity = np.ones(len(unique_ids), dtype=float)
    return within_rank_correlation_arrays(
        joint[value_columns[0]].to_numpy(dtype=float),
        joint[value_columns[1]].to_numpy(dtype=float),
        codes,
        multiplicity,
        cluster_sizes_local,
    )


def deterministic_seed(*parts: str) -> int:
    token = '|'.join(parts).encode('utf-8')
    return int(hashlib.sha256(token).hexdigest()[:8], 16) ^ SEED


# One coherent participant bootstrap is shared by every pair. This preserves the
# covariance among pair estimates and allows the family-level contrast to be built
# without a second, prohibitively expensive bootstrap.
GLOBAL_PARTICIPANTS = analysis['participant_id'].astype('string').drop_duplicates().to_numpy()
GLOBAL_PARTICIPANT_INDEX = {
    participant: index for index, participant in enumerate(GLOBAL_PARTICIPANTS)
}
N_COHERENT_BOOTSTRAP = max(N_CORRELATION_BOOTSTRAP, N_FAMILY_BOOTSTRAP)
bootstrap_count_path = MODEL_DIR / f'coherent_participant_bootstrap_counts_{N_COHERENT_BOOTSTRAP}.npz'
if bootstrap_count_path.exists():
    COHERENT_BOOTSTRAP_COUNTS = np.load(bootstrap_count_path)['counts']
else:
    bootstrap_generator = np.random.default_rng(
        deterministic_seed('coherent', 'participant', 'bootstrap')
    )
    COHERENT_BOOTSTRAP_COUNTS = bootstrap_generator.multinomial(
        len(GLOBAL_PARTICIPANTS),
        np.full(len(GLOBAL_PARTICIPANTS), 1.0 / len(GLOBAL_PARTICIPANTS)),
        size=N_COHERENT_BOOTSTRAP,
    ).astype(np.int16)
    np.savez_compressed(bootstrap_count_path, counts=COHERENT_BOOTSTRAP_COUNTS)


def prepare_between_arrays(profiles: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    participant_codes = np.asarray([
        GLOBAL_PARTICIPANT_INDEX[str(participant)] for participant in profiles.index
    ], dtype=int)
    return (
        profiles.iloc[:, 0].to_numpy(dtype=float),
        profiles.iloc[:, 1].to_numpy(dtype=float),
        participant_codes,
    )


def prepare_within_arrays(joint: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    value_columns = [column for column in joint.columns if column != 'participant_id']
    participant_codes = joint['participant_id'].astype('string').map(
        GLOBAL_PARTICIPANT_INDEX
    ).to_numpy(dtype=int)
    cluster_sizes = np.bincount(
        participant_codes, minlength=len(GLOBAL_PARTICIPANTS)
    ).astype(float)
    return (
        joint[value_columns[0]].to_numpy(dtype=float),
        joint[value_columns[1]].to_numpy(dtype=float),
        participant_codes,
        cluster_sizes,
    )


def coherent_pair_bootstrap(
    feature_x: str,
    feature_y: str,
    level: str,
    estimator_frame: pd.DataFrame,
) -> np.ndarray:
    safe_name = (
        f'{FEATURE_CODES[feature_x]}_{FEATURE_CODES[feature_y]}_{level}_'
        f'coherent_{N_COHERENT_BOOTSTRAP}.npz'
    )
    checkpoint = BOOTSTRAP_DIR / safe_name
    if checkpoint.exists():
        stored = np.load(checkpoint)['draws']
        if len(stored) == N_COHERENT_BOOTSTRAP:
            return stored

    draws = np.full(N_COHERENT_BOOTSTRAP, np.nan, dtype=float)
    if level == 'between':
        x, y, participant_codes = prepare_between_arrays(estimator_frame)
        for replicate, counts in enumerate(COHERENT_BOOTSTRAP_COUNTS):
            frequencies = counts[participant_codes]
            draws[replicate] = weighted_spearman_with_frequencies(x, y, frequencies)
    elif level == 'within':
        x, y, participant_codes, cluster_sizes = prepare_within_arrays(estimator_frame)
        for replicate, counts in enumerate(COHERENT_BOOTSTRAP_COUNTS):
            draws[replicate] = within_rank_correlation_arrays(
                x, y, participant_codes, counts, cluster_sizes
            )
    else:
        raise ValueError(level)
    np.savez_compressed(checkpoint, draws=draws)
    return draws


correlation_rows = []
PAIR_BOOTSTRAP_DRAWS: dict[tuple[str, str, str], np.ndarray] = {}
for pair_index, support in pair_support.iterrows():
    feature_x = support['feature_x']
    feature_y = support['feature_y']
    row = support.to_dict()

    if support['between_gate']:
        between_frame = prepare_between_profiles(analysis, feature_x, feature_y, summary='mean')
        between_estimate = between_spearman(between_frame)
        between_draws_all = coherent_pair_bootstrap(
            feature_x, feature_y, 'between', between_frame
        )
        PAIR_BOOTSTRAP_DRAWS[(feature_x, feature_y, 'between')] = between_draws_all
        between_draws = between_draws_all[:N_CORRELATION_BOOTSTRAP]
        valid_between = between_draws[np.isfinite(between_draws)]
        between_low, between_high = percentile_interval(valid_between)
        between_success = len(valid_between) / N_CORRELATION_BOOTSTRAP
    else:
        between_estimate = between_low = between_high = between_success = np.nan

    if support['within_gate']:
        within_frame = prepare_within_pair(analysis, feature_x, feature_y)
        within_estimate = within_rank_correlation(within_frame)
        within_draws_all = coherent_pair_bootstrap(
            feature_x, feature_y, 'within', within_frame
        )
        PAIR_BOOTSTRAP_DRAWS[(feature_x, feature_y, 'within')] = within_draws_all
        within_draws = within_draws_all[:N_CORRELATION_BOOTSTRAP]
        valid_within = within_draws[np.isfinite(within_draws)]
        within_low, within_high = percentile_interval(valid_within)
        within_success = len(valid_within) / N_CORRELATION_BOOTSTRAP
    else:
        within_estimate = within_low = within_high = within_success = np.nan

    row.update({
        'between_rho': between_estimate,
        'between_ci95_low': between_low,
        'between_ci95_high': between_high,
        'between_bootstrap_success': between_success,
        'within_rho': within_estimate,
        'within_ci95_low': within_low,
        'within_ci95_high': within_high,
        'within_bootstrap_success': within_success,
    })
    correlation_rows.append(row)
    if (pair_index + 1) % 20 == 0 or pair_index + 1 == len(pair_support):
        print(f'Completed {pair_index + 1}/{len(pair_support)} pairs')

pairwise_correlations = pd.DataFrame(correlation_rows)
pairwise_correlations['between_ci_valid'] = (
    pairwise_correlations['between_bootstrap_success'].ge(MIN_BOOTSTRAP_SUCCESS)
)
pairwise_correlations['within_ci_valid'] = (
    pairwise_correlations['within_bootstrap_success'].ge(MIN_BOOTSTRAP_SUCCESS)
)
export_dataframe(pairwise_correlations, TABLE_DIR / 'pairwise_correlations_with_ci.csv')
display(pairwise_correlations.head(20))


## 9. Correlation matrices and structured overlap classification

Potential redundancy is never assigned from a correlation threshold alone. Known same-ledger relationships are identified first. Non-ledger pairs require strong structure at both levels to be considered potentially redundant. Discordance between levels is retained as a context-dependent result rather than averaged away.


In [ ]:
feature_lookup = FEATURE_REGISTRY.set_index('feature')
feature_order = PAIRWISE_FEATURES


def correlation_matrix(frame: pd.DataFrame, value_column: str) -> pd.DataFrame:
    matrix = pd.DataFrame(np.nan, index=feature_order, columns=feature_order, dtype=float)
    np.fill_diagonal(matrix.values, 1.0)
    for row in frame.itertuples(index=False):
        value = getattr(row, value_column)
        matrix.loc[row.feature_x, row.feature_y] = value
        matrix.loc[row.feature_y, row.feature_x] = value
    return matrix


between_matrix = correlation_matrix(pairwise_correlations, 'between_rho')
within_matrix = correlation_matrix(pairwise_correlations, 'within_rho')


def minimum_same_sign_ci_magnitude(low: float, high: float) -> float:
    if not np.isfinite(low) or not np.isfinite(high) or low <= 0 <= high:
        return 0.0
    return float(min(abs(low), abs(high)))


def classify_pair(row: pd.Series) -> str:
    if bool(row['structural_dependency']):
        return 'structurally dependent'
    between = row['between_rho']
    within = row['within_rho']
    if not np.isfinite(between) or not np.isfinite(within):
        return 'insufficient joint support'
    abs_between, abs_within = abs(between), abs(within)
    same_sign = np.sign(between) == np.sign(within)
    between_ci_strength = minimum_same_sign_ci_magnitude(
        row['between_ci95_low'], row['between_ci95_high']
    )
    within_ci_strength = minimum_same_sign_ci_magnitude(
        row['within_ci95_low'], row['within_ci95_high']
    )
    stable_cis = bool(row['between_ci_valid'] and row['within_ci_valid'])
    if (
        abs_between >= 0.80 and abs_within >= 0.80 and same_sign and stable_cis
        and between_ci_strength >= 0.60 and within_ci_strength >= 0.60
    ):
        return 'potentially redundant'
    if (
        abs_between >= 0.60 and abs_within >= 0.60 and same_sign and stable_cis
        and between_ci_strength >= 0.35 and within_ci_strength >= 0.35
    ):
        return 'strongly overlapping'
    if abs_between - abs_within >= 0.30 and abs_between >= 0.40:
        return 'between-participant dominant'
    if abs_within - abs_between >= 0.30 and abs_within >= 0.40:
        return 'within-participant dominant'
    if max(abs_between, abs_within) >= 0.40:
        return 'overlapping or context dependent'
    if abs_between < 0.30 and abs_within < 0.30:
        return 'largely distinct'
    return 'weak-to-moderate overlap'


pairwise_correlations['relationship_class'] = pairwise_correlations.apply(classify_pair, axis=1)
pairwise_correlations['absolute_between_within_difference'] = (
    pairwise_correlations['between_rho'].abs() - pairwise_correlations['within_rho'].abs()
).abs()
pairwise_correlations['same_sign'] = (
    np.sign(pairwise_correlations['between_rho'])
    == np.sign(pairwise_correlations['within_rho'])
)

relationship_summary = (
    pairwise_correlations['relationship_class'].value_counts(dropna=False)
    .rename_axis('relationship_class').reset_index(name='pair_count')
)

export_dataframe(between_matrix.reset_index(names='feature'), TABLE_DIR / 'between_participant_correlation_matrix.csv')
export_dataframe(within_matrix.reset_index(names='feature'), TABLE_DIR / 'within_participant_correlation_matrix.csv')
export_dataframe(pairwise_correlations, TABLE_DIR / 'pairwise_relationship_classification.csv')
export_dataframe(relationship_summary, TABLE_DIR / 'relationship_classification_summary.csv')

display(relationship_summary)
display(
    pairwise_correlations.sort_values(
        ['structural_dependency', 'absolute_between_within_difference'],
        ascending=[False, False],
    ).head(25)
)


## 10. Family organization: bootstrap contrast and label-permutation sensitivity

The contrast is the mean absolute within-family correlation minus the mean absolute between-family correlation. Structurally dependent pairs are excluded. The primary uncertainty interval resamples participants and reconstructs the complete matrix. A supplementary label permutation preserves family sizes and the observed support mask.


In [ ]:
NONSTRUCTURAL_PAIRS = pairwise_correlations.loc[
    ~pairwise_correlations['structural_dependency']
].copy()


def family_contrast(pair_frame: pd.DataFrame, value_col: str) -> dict[str, float]:
    valid = pair_frame.loc[np.isfinite(pair_frame[value_col])].copy()
    within_values = valid.loc[valid['same_family'], value_col].abs()
    between_values = valid.loc[~valid['same_family'], value_col].abs()
    if within_values.empty or between_values.empty:
        return {
            'within_family_mean_abs_rho': np.nan,
            'between_family_mean_abs_rho': np.nan,
            'delta': np.nan,
            'n_within_pairs': len(within_values),
            'n_between_pairs': len(between_values),
        }
    return {
        'within_family_mean_abs_rho': float(within_values.mean()),
        'between_family_mean_abs_rho': float(between_values.mean()),
        'delta': float(within_values.mean() - between_values.mean()),
        'n_within_pairs': len(within_values),
        'n_between_pairs': len(between_values),
    }


observed_family_rows = []
for level, value_col in [('between', 'between_rho'), ('within', 'within_rho')]:
    observed_family_rows.append({'level': level, **family_contrast(NONSTRUCTURAL_PAIRS, value_col)})
observed_family_contrast = pd.DataFrame(observed_family_rows)

# The pairwise bootstrap used one common participant resample for every pair.
# Family-level bootstrap contrasts can therefore be calculated coherently from
# those stored draws without rerunning the full correlation estimator.
family_bootstrap_rows = []
for level in ['between', 'within']:
    eligible_pairs = NONSTRUCTURAL_PAIRS.loc[
        NONSTRUCTURAL_PAIRS[f'{level}_gate']
    ].copy()
    for replicate in range(N_FAMILY_BOOTSTRAP):
        values = []
        for pair in eligible_pairs.itertuples(index=False):
            key = (pair.feature_x, pair.feature_y, level)
            draws = PAIR_BOOTSTRAP_DRAWS.get(key)
            values.append(draws[replicate] if draws is not None else np.nan)
        sample = eligible_pairs.copy()
        sample['bootstrap_value'] = values
        result = family_contrast(sample, 'bootstrap_value')
        family_bootstrap_rows.append({
            'replicate': replicate,
            'level': level,
            **result,
        })
family_bootstrap = pd.DataFrame(family_bootstrap_rows)

family_summary_rows = []
for level in ['between', 'within']:
    observed = observed_family_contrast.loc[
        observed_family_contrast['level'] == level, 'delta'
    ].iloc[0]
    draws = family_bootstrap.loc[
        family_bootstrap['level'] == level, 'delta'
    ].dropna()
    low, high = percentile_interval(draws)
    family_summary_rows.append({
        'level': level,
        'delta': observed,
        'ci95_low': low,
        'ci95_high': high,
        'bootstrap_success': len(draws) / N_FAMILY_BOOTSTRAP,
    })
family_contrast_summary = pd.DataFrame(family_summary_rows)

# Permute family assignments while preserving family sizes and the observed support mask.
family_by_feature = feature_lookup.loc[PAIRWISE_FEATURES, 'family'].to_dict()
original_labels = np.asarray([family_by_feature[feature] for feature in PAIRWISE_FEATURES])
permutation_generator = np.random.default_rng(deterministic_seed('family', 'permutation'))
permutation_rows = []
for level, value_col in [('between', 'between_rho'), ('within', 'within_rho')]:
    observed = observed_family_contrast.loc[
        observed_family_contrast['level'] == level, 'delta'
    ].iloc[0]
    null_values = np.full(N_FAMILY_PERMUTATIONS, np.nan)
    base_pairs = NONSTRUCTURAL_PAIRS.loc[np.isfinite(NONSTRUCTURAL_PAIRS[value_col])].copy()
    values = base_pairs[value_col].abs().to_numpy()
    for replicate in range(N_FAMILY_PERMUTATIONS):
        permuted = permutation_generator.permutation(original_labels)
        mapping = dict(zip(PAIRWISE_FEATURES, permuted))
        same = np.fromiter(
            (mapping[x] == mapping[y] for x, y in zip(base_pairs['feature_x'], base_pairs['feature_y'])),
            dtype=bool,
            count=len(base_pairs),
        )
        if same.any() and (~same).any():
            null_values[replicate] = values[same].mean() - values[~same].mean()
    valid_null = null_values[np.isfinite(null_values)]
    p_value = (1 + np.sum(valid_null >= observed)) / (1 + len(valid_null))
    permutation_rows.append({
        'level': level,
        'observed_delta': observed,
        'permutation_p_value': p_value,
        'n_valid_permutations': len(valid_null),
        'null_mean': float(np.mean(valid_null)),
        'null_sd': float(np.std(valid_null, ddof=1)),
    })
    export_dataframe(
        pd.DataFrame({'permuted_delta': valid_null}),
        MODEL_DIR / f'family_permutation_{level}.csv',
    )

family_permutation_summary = pd.DataFrame(permutation_rows)
family_contrast_summary = family_contrast_summary.merge(
    family_permutation_summary, on='level', how='left'
)

export_dataframe(observed_family_contrast, TABLE_DIR / 'family_contrast_observed.csv')
export_dataframe(family_bootstrap, MODEL_DIR / 'family_contrast_bootstrap.csv')
export_dataframe(family_contrast_summary, TABLE_DIR / 'family_contrast_summary.csv')
display(family_contrast_summary)


## 11. Multilevel PCA: complete-case audit, parallel analysis, and bootstrap stability

PCA is descriptive and is not used to form a quality score. The primary eight-feature set contains two QADD observables, three QGAIN timescales, one high-coverage QREV comparator, and two complementary QCHAN observables. Known same-ledger and sparse outputs remain outside the primary PCA. Between- and within-participant correlation structures are decomposed separately.


In [ ]:
def safe_correlation_from_covariance(covariance: np.ndarray) -> np.ndarray:
    diagonal = np.diag(covariance)
    standard_deviation = np.sqrt(np.clip(diagonal, 0, None))
    denominator = np.outer(standard_deviation, standard_deviation)
    correlation = np.divide(
        covariance, denominator,
        out=np.zeros_like(covariance, dtype=float),
        where=denominator > 0,
    )
    np.fill_diagonal(correlation, 1.0)
    return correlation


def eigendecompose_correlation(correlation: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    eigenvalues, eigenvectors = np.linalg.eigh(correlation)
    order = np.argsort(eigenvalues)[::-1]
    return eigenvalues[order], eigenvectors[:, order]


def between_pca_input(data: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    z_columns = [f'{feature}__z' for feature in features]
    profiles = data.groupby('participant_id', observed=True)[z_columns].mean().dropna()
    profiles.columns = features
    return profiles


def within_pca_correlation(data: pd.DataFrame, features: list[str]) -> tuple[np.ndarray, pd.DataFrame]:
    z_columns = [f'{feature}__z' for feature in features]
    complete = data[['participant_id'] + z_columns].dropna().copy()
    counts = complete.groupby('participant_id', observed=True).size()
    complete = complete.loc[complete['participant_id'].isin(counts[counts >= 2].index)].copy()
    covariance_matrices = []
    for _, participant_data in complete.groupby('participant_id', observed=True):
        values = participant_data[z_columns].to_numpy(dtype=float)
        if len(values) >= 2:
            covariance_matrices.append(np.cov(values, rowvar=False, ddof=1))
    if not covariance_matrices:
        raise ValueError('No repeated complete participants for within-PCA.')
    average_covariance = np.mean(covariance_matrices, axis=0)
    return safe_correlation_from_covariance(average_covariance), complete


def align_loadings(reference: np.ndarray, candidate: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    similarity = np.abs(reference.T @ candidate)
    reference_indices, candidate_indices = linear_sum_assignment(-similarity)
    aligned = np.zeros_like(candidate)
    component_similarity = np.full(reference.shape[1], np.nan)
    for reference_index, candidate_index in zip(reference_indices, candidate_indices):
        vector = candidate[:, candidate_index]
        sign = 1.0 if np.dot(reference[:, reference_index], vector) >= 0 else -1.0
        aligned[:, reference_index] = sign * vector
        component_similarity[reference_index] = abs(
            np.dot(reference[:, reference_index], aligned[:, reference_index])
        )
    return aligned, component_similarity


def parallel_between(n_rows: int, n_features: int, n_iterations: int, seed: int) -> np.ndarray:
    generator = np.random.default_rng(seed)
    eigenvalues = np.empty((n_iterations, n_features))
    for index in range(n_iterations):
        random_data = generator.normal(size=(n_rows, n_features))
        eigenvalues[index], _ = eigendecompose_correlation(np.corrcoef(random_data, rowvar=False))
    return eigenvalues


def parallel_within(
    participant_sizes: list[int],
    n_features: int,
    n_iterations: int,
    seed: int,
) -> np.ndarray:
    generator = np.random.default_rng(seed)
    eigenvalues = np.empty((n_iterations, n_features))
    for index in range(n_iterations):
        covariances = []
        for size in participant_sizes:
            values = generator.normal(size=(size, n_features))
            covariances.append(np.cov(values, rowvar=False, ddof=1))
        correlation = safe_correlation_from_covariance(np.mean(covariances, axis=0))
        eigenvalues[index], _ = eigendecompose_correlation(correlation)
    return eigenvalues


# Primary between PCA.
between_profiles = between_pca_input(analysis, PRIMARY_PCA_FEATURES)
between_correlation = between_profiles.corr().to_numpy(dtype=float)
between_eigenvalues, between_loadings = eigendecompose_correlation(between_correlation)

# Primary within PCA.
within_correlation_pca, within_complete = within_pca_correlation(analysis, PRIMARY_PCA_FEATURES)
within_eigenvalues, within_loadings = eigendecompose_correlation(within_correlation_pca)
within_sizes = within_complete.groupby('participant_id', observed=True).size().tolist()

between_parallel = parallel_between(
    len(between_profiles), len(PRIMARY_PCA_FEATURES), N_PARALLEL_ANALYSIS,
    deterministic_seed('pca', 'parallel', 'between'),
)
within_parallel = parallel_within(
    within_sizes, len(PRIMARY_PCA_FEATURES), N_PARALLEL_ANALYSIS,
    deterministic_seed('pca', 'parallel', 'within'),
)
between_parallel95 = np.quantile(between_parallel, 0.95, axis=0)
within_parallel95 = np.quantile(within_parallel, 0.95, axis=0)

# Bootstrap stability.
pca_bootstrap_rows = []
between_loading_draws = []
within_loading_draws = []

pca_generator = np.random.default_rng(deterministic_seed('pca', 'bootstrap'))
between_values = between_profiles.to_numpy(dtype=float)
within_covariances = []
for _, frame in within_complete.groupby('participant_id', observed=True):
    values = frame[[f'{f}__z' for f in PRIMARY_PCA_FEATURES]].to_numpy(dtype=float)
    within_covariances.append(np.cov(values, rowvar=False, ddof=1))
within_covariances = np.asarray(within_covariances)
within_participant_indices = np.arange(len(within_covariances))

for replicate in range(N_PCA_BOOTSTRAP):
    # Between participants.
    sample_indices = pca_generator.integers(0, len(between_values), size=len(between_values))
    sample = between_values[sample_indices]
    sample_correlation = np.corrcoef(sample, rowvar=False)
    sample_eigenvalues, sample_loadings = eigendecompose_correlation(sample_correlation)
    aligned_between, similarity_between = align_loadings(between_loadings, sample_loadings)
    between_loading_draws.append(aligned_between)
    for component in range(len(PRIMARY_PCA_FEATURES)):
        pca_bootstrap_rows.append({
            'replicate': replicate, 'level': 'between', 'component': component + 1,
            'eigenvalue': sample_eigenvalues[component],
            'loading_similarity': similarity_between[component],
        })

    # Within participants.
    sampled_indices = pca_generator.choice(
        within_participant_indices, size=len(within_participant_indices), replace=True
    )
    sample_within_correlation = safe_correlation_from_covariance(
        within_covariances[sampled_indices].mean(axis=0)
    )
    sample_eigenvalues, sample_loadings = eigendecompose_correlation(sample_within_correlation)
    aligned_within, similarity_within = align_loadings(within_loadings, sample_loadings)
    within_loading_draws.append(aligned_within)
    for component in range(len(PRIMARY_PCA_FEATURES)):
        pca_bootstrap_rows.append({
            'replicate': replicate, 'level': 'within', 'component': component + 1,
            'eigenvalue': sample_eigenvalues[component],
            'loading_similarity': similarity_within[component],
        })

    if (replicate + 1) % max(20, N_PCA_BOOTSTRAP // 10) == 0:
        print(f'PCA bootstrap {replicate + 1}/{N_PCA_BOOTSTRAP}')

pca_bootstrap = pd.DataFrame(pca_bootstrap_rows)
between_loading_draws = np.asarray(between_loading_draws)
within_loading_draws = np.asarray(within_loading_draws)

pca_component_rows = []
for level, eigenvalues, parallel95, loading_draws in [
    ('between', between_eigenvalues, between_parallel95, between_loading_draws),
    ('within', within_eigenvalues, within_parallel95, within_loading_draws),
]:
    for component in range(len(PRIMARY_PCA_FEATURES)):
        similarities = pca_bootstrap.loc[
            (pca_bootstrap['level'] == level)
            & (pca_bootstrap['component'] == component + 1),
            'loading_similarity',
        ].dropna()
        median_similarity = float(similarities.median())
        p05_similarity = float(similarities.quantile(0.05))
        retained = bool(
            eigenvalues[component] > parallel95[component]
            and median_similarity >= PCA_LOADING_STABILITY_MEDIAN
            and p05_similarity >= PCA_LOADING_STABILITY_P05
        )
        pca_component_rows.append({
            'level': level,
            'component': component + 1,
            'eigenvalue': eigenvalues[component],
            'variance_explained_percent': 100 * eigenvalues[component] / len(PRIMARY_PCA_FEATURES),
            'parallel_95th_eigenvalue': parallel95[component],
            'bootstrap_loading_similarity_median': median_similarity,
            'bootstrap_loading_similarity_p05': p05_similarity,
            'retained_stable_component': retained,
        })

pca_components = pd.DataFrame(pca_component_rows)

pca_loading_rows = []
for level, reference_loadings, loading_draws in [
    ('between', between_loadings, between_loading_draws),
    ('within', within_loadings, within_loading_draws),
]:
    for feature_index, feature in enumerate(PRIMARY_PCA_FEATURES):
        for component in range(len(PRIMARY_PCA_FEATURES)):
            draws = loading_draws[:, feature_index, component]
            low, high = percentile_interval(draws)
            pca_loading_rows.append({
                'level': level,
                'feature': feature,
                'feature_code': FEATURE_CODES[feature],
                'component': component + 1,
                'loading': reference_loadings[feature_index, component],
                'ci95_low': low,
                'ci95_high': high,
                'retained_stable_component': bool(
                    pca_components.loc[
                        (pca_components['level'] == level)
                        & (pca_components['component'] == component + 1),
                        'retained_stable_component',
                    ].iloc[0]
                ),
            })

pca_loadings = pd.DataFrame(pca_loading_rows)

# Complete-case inclusion audit by cohort.
pca_complete_case_audit = pd.DataFrame([
    {
        'level': 'between',
        'complete_participants': len(between_profiles),
        'excluded_participants': analysis['participant_id'].nunique() - len(between_profiles),
        'complete_recordings': np.nan,
        'repeated_complete_participants': np.nan,
    },
    {
        'level': 'within',
        'complete_participants': within_complete['participant_id'].nunique(),
        'excluded_participants': analysis['participant_id'].nunique() - within_complete['participant_id'].nunique(),
        'complete_recordings': len(within_complete),
        'repeated_complete_participants': within_complete['participant_id'].nunique(),
    },
])

for frame, filename in [
    (pca_components, 'pca_components_and_retention.csv'),
    (pca_loadings, 'pca_loadings_with_bootstrap_ci.csv'),
    (pca_bootstrap, 'pca_bootstrap_stability.csv'),
    (pca_complete_case_audit, 'pca_complete_case_audit.csv'),
]:
    export_dataframe(frame, TABLE_DIR / filename if 'bootstrap' not in filename else MODEL_DIR / filename)

export_dataframe(
    pd.DataFrame(between_parallel, columns=[f'PC{index + 1}' for index in range(len(PRIMARY_PCA_FEATURES))]),
    MODEL_DIR / 'pca_parallel_between.csv',
)
export_dataframe(
    pd.DataFrame(within_parallel, columns=[f'PC{index + 1}' for index in range(len(PRIMARY_PCA_FEATURES))]),
    MODEL_DIR / 'pca_parallel_within.csv',
)

display(pca_components)
display(pca_loadings.loc[pca_loadings['retained_stable_component']])


## 12. Prespecified sensitivity analyses

Sensitivities address the participant summary, cohort composition, longitudinal time, frozen transformations, QCHAN truncation, conditional QREV support, and equal-participant within-session sampling. They do not replace the primary matrices.


In [ ]:
sensitivity_rows = []

# Between-participant participant-median sensitivity.
for pair in pairwise_correlations.itertuples(index=False):
    median_profiles = prepare_between_profiles(
        analysis, pair.feature_x, pair.feature_y, summary='median'
    )
    median_rho = between_spearman(median_profiles) if len(median_profiles) >= MIN_BETWEEN_PARTICIPANTS else np.nan
    sensitivity_rows.append({
        'sensitivity': 'participant_median_between',
        'feature_x': pair.feature_x,
        'feature_y': pair.feature_y,
        'estimate': median_rho,
        'primary_estimate': pair.between_rho,
        'difference': median_rho - pair.between_rho if np.isfinite(median_rho) else np.nan,
    })

# Monotonic frozen transformations should not change rank-based correlations.
for pair in pairwise_correlations.itertuples(index=False):
    raw_joint = analysis[['participant_id', pair.feature_x, pair.feature_y]].dropna().copy()
    raw_profiles = raw_joint.groupby('participant_id', observed=True)[[pair.feature_x, pair.feature_y]].mean()
    raw_between = between_spearman(raw_profiles) if len(raw_profiles) >= MIN_BETWEEN_PARTICIPANTS else np.nan
    sensitivity_rows.append({
        'sensitivity': 'raw_value_between_rank',
        'feature_x': pair.feature_x,
        'feature_y': pair.feature_y,
        'estimate': raw_between,
        'primary_estimate': pair.between_rho,
        'difference': raw_between - pair.between_rho if np.isfinite(raw_between) else np.nan,
    })

# ALS-only and control-only matrices.
subset_correlation_rows = []
for cohort in ['ALS', 'CONTROL']:
    subset = analysis.loc[analysis['cohort'] == cohort].copy()
    for pair in pairwise_correlations.itertuples(index=False):
        between_frame = prepare_between_profiles(subset, pair.feature_x, pair.feature_y)
        between_value = between_spearman(between_frame) if len(between_frame) >= MIN_BETWEEN_PARTICIPANTS else np.nan
        within_frame = prepare_within_pair(subset, pair.feature_x, pair.feature_y)
        within_value = (
            within_rank_correlation(within_frame)
            if len(within_frame) >= MIN_WITHIN_JOINT_RECORDINGS
            and within_frame['participant_id'].nunique() >= MIN_WITHIN_REPEATED_PARTICIPANTS
            else np.nan
        )
        subset_correlation_rows.append({
            'cohort': cohort,
            'feature_x': pair.feature_x,
            'feature_y': pair.feature_y,
            'between_rho': between_value,
            'n_between_participants': len(between_frame),
            'within_rho': within_value,
            'n_within_recordings': len(within_frame),
            'n_within_participants': within_frame['participant_id'].nunique() if not within_frame.empty else 0,
        })
subset_correlations = pd.DataFrame(subset_correlation_rows)

# Time-adjusted ALS participant profiles using a common within-participant slope.
time_adjusted_profiles = {}
als_data = analysis.loc[analysis['cohort'] == 'ALS'].copy()
for feature in PAIRWISE_FEATURES:
    value_col = f'{feature}__z'
    d = als_data[['participant_id', 'time_since_first_years', value_col]].dropna().copy()
    d['_time_centered'] = d['time_since_first_years'] - d.groupby('participant_id')['time_since_first_years'].transform('mean')
    d['_value_centered'] = d[value_col] - d.groupby('participant_id')[value_col].transform('mean')
    denominator = float(np.sum(d['_time_centered'] ** 2))
    slope = float(np.sum(d['_time_centered'] * d['_value_centered']) / denominator) if denominator > 0 else 0.0
    d['_adjusted'] = d[value_col] - slope * d['time_since_first_years']
    time_adjusted_profiles[feature] = d.groupby('participant_id')['_adjusted'].mean()
time_adjusted_frame = pd.DataFrame(time_adjusted_profiles)
time_adjusted_rows = []
for pair in pairwise_correlations.itertuples(index=False):
    pair_frame = time_adjusted_frame[[pair.feature_x, pair.feature_y]].dropna()
    estimate = between_spearman(pair_frame) if len(pair_frame) >= MIN_BETWEEN_PARTICIPANTS else np.nan
    time_adjusted_rows.append({
        'feature_x': pair.feature_x, 'feature_y': pair.feature_y,
        'time_adjusted_als_between_rho': estimate,
        'n_participants': len(pair_frame),
    })
time_adjusted_correlations = pd.DataFrame(time_adjusted_rows)

# Random two-recording change sensitivity for within-participant structure.
# This is a sensitivity analysis only; it preserves two real sessions per participant
# and does not replace the primary residualized-rank estimator.
random_pair_rows = []
pair_generator = np.random.default_rng(deterministic_seed('random', 'recording_pairs'))
for pair in pairwise_correlations.itertuples(index=False):
    if not pair.within_gate:
        random_pair_rows.append({
            'feature_x': pair.feature_x,
            'feature_y': pair.feature_y,
            'median_random_pair_change_rho': np.nan,
            'p05': np.nan,
            'p95': np.nan,
            'n_valid_resamples': 0,
        })
        continue
    x_col = f'{pair.feature_x}__transformed'
    y_col = f'{pair.feature_y}__transformed'
    joint = analysis[['participant_id', 'recording_date', x_col, y_col]].dropna().copy()
    blocks = []
    for _, participant_data in joint.groupby('participant_id', observed=True):
        if len(participant_data) < 2:
            continue
        ordered = participant_data.sort_values('recording_date')
        blocks.append((
            ordered[x_col].to_numpy(dtype=float),
            ordered[y_col].to_numpy(dtype=float),
        ))
    estimates = np.full(N_RANDOM_PAIR_RESAMPLES, np.nan)
    for replicate in range(N_RANDOM_PAIR_RESAMPLES):
        changes_x = []
        changes_y = []
        for values_x, values_y in blocks:
            chosen = np.sort(pair_generator.choice(len(values_x), size=2, replace=False))
            changes_x.append(values_x[chosen[1]] - values_x[chosen[0]])
            changes_y.append(values_y[chosen[1]] - values_y[chosen[0]])
        if len(changes_x) >= MIN_WITHIN_REPEATED_PARTICIPANTS:
            estimates[replicate] = stats.spearmanr(changes_x, changes_y).statistic
    valid_estimates = estimates[np.isfinite(estimates)]
    random_pair_rows.append({
        'feature_x': pair.feature_x,
        'feature_y': pair.feature_y,
        'median_random_pair_change_rho': float(np.median(valid_estimates)) if len(valid_estimates) else np.nan,
        'p05': float(np.quantile(valid_estimates, 0.05)) if len(valid_estimates) else np.nan,
        'p95': float(np.quantile(valid_estimates, 0.95)) if len(valid_estimates) else np.nan,
        'n_valid_resamples': len(valid_estimates),
    })
random_pair_sensitivity = pd.DataFrame(random_pair_rows)

# Signed QCHAN precursor sensitivity.
signed_qchan_map = {
    'qchan_rolloff95_deficit_hz': 'qchan_rolloff95_signed_difference_hz',
    'qchan_highband_ratio_deficit': 'qchan_highband_ratio_signed_difference',
    'qchan_tilt_steepening_db_per_oct': 'qchan_tilt_signed_difference_db_per_oct',
}
signed_qchan_rows = []
for truncated, signed in signed_qchan_map.items():
    if signed not in analysis.columns:
        continue
    for other in PAIRWISE_FEATURES:
        if other == truncated:
            continue
        joint = analysis[['participant_id', signed, f'{other}__transformed']].dropna().copy()
        profiles = joint.groupby('participant_id')[[signed, f'{other}__transformed']].mean().dropna()
        between_value = between_spearman(profiles) if len(profiles) >= MIN_BETWEEN_PARTICIPANTS else np.nan
        signed_qchan_rows.append({
            'truncated_feature': truncated,
            'signed_precursor': signed,
            'other_feature': other,
            'signed_between_rho': between_value,
        })
signed_qchan_sensitivity = pd.DataFrame(signed_qchan_rows)

# Conditional-QREV exclusion and core-only family contrasts.
high_support_features = structural_eligibility.loc[
    structural_eligibility['include_pairwise']
    & structural_eligibility['n_available_participants'].ge(100),
    'feature',
].tolist()
core_features = structural_eligibility.loc[
    structural_eligibility['include_pairwise']
    & structural_eligibility['inference_tier'].eq('core'),
    'feature',
].tolist()
structure_set_rows = []
for set_name, set_features in [
    ('high_support', high_support_features),
    ('core_only', core_features),
]:
    set_pairs = NONSTRUCTURAL_PAIRS.loc[
        NONSTRUCTURAL_PAIRS['feature_x'].isin(set_features)
        & NONSTRUCTURAL_PAIRS['feature_y'].isin(set_features)
    ]
    for level, value_col in [('between', 'between_rho'), ('within', 'within_rho')]:
        structure_set_rows.append({
            'feature_set': set_name,
            'level': level,
            **family_contrast(set_pairs, value_col),
        })
structure_set_sensitivity = pd.DataFrame(structure_set_rows)

# PCA feature-set sensitivities without using them to redefine the primary PCA.
pca_sensitivity_rows = []
for set_name, features in PCA_SENSITIVITY_FEATURES.items():
    profiles = between_pca_input(analysis, features)
    correlation = profiles.corr().to_numpy(dtype=float)
    eigenvalues, _ = eigendecompose_correlation(correlation)
    within_correlation, within_data = within_pca_correlation(analysis, features)
    within_eigenvalues_sensitivity, _ = eigendecompose_correlation(within_correlation)
    for component, value in enumerate(eigenvalues, start=1):
        pca_sensitivity_rows.append({
            'feature_set': set_name, 'level': 'between', 'component': component,
            'eigenvalue': value, 'n_participants': len(profiles),
        })
    for component, value in enumerate(within_eigenvalues_sensitivity, start=1):
        pca_sensitivity_rows.append({
            'feature_set': set_name, 'level': 'within', 'component': component,
            'eigenvalue': value,
            'n_participants': within_data['participant_id'].nunique(),
        })
pca_sensitivity = pd.DataFrame(pca_sensitivity_rows)

for frame, filename in [
    (pd.DataFrame(sensitivity_rows), 'pairwise_summary_sensitivities.csv'),
    (subset_correlations, 'cohort_stratified_correlations.csv'),
    (time_adjusted_correlations, 'als_time_adjusted_between_correlations.csv'),
    (random_pair_sensitivity, 'random_recording_pair_within_sensitivity.csv'),
    (signed_qchan_sensitivity, 'signed_qchan_sensitivity.csv'),
    (structure_set_sensitivity, 'feature_set_family_contrast_sensitivity.csv'),
    (pca_sensitivity, 'pca_feature_set_sensitivity.csv'),
]:
    export_dataframe(frame, TABLE_DIR / filename)

sensitivity_summary = pd.DataFrame([
    {
        'sensitivity': 'participant mean versus median',
        'maximum_absolute_correlation_difference': pd.DataFrame(sensitivity_rows)
            .loc[lambda x: x['sensitivity'] == 'participant_median_between', 'difference'].abs().max(),
    },
    {
        'sensitivity': 'frozen transformed participant means versus raw participant means',
        'maximum_absolute_correlation_difference': pd.DataFrame(sensitivity_rows)
            .loc[lambda x: x['sensitivity'] == 'raw_value_between_rank', 'difference'].abs().max(),
    },
])
export_dataframe(sensitivity_summary, TABLE_DIR / 'sensitivity_summary.csv')
display(sensitivity_summary)


## 13. Main figure source data and publication figure

The main figure uses compact feature codes to avoid text overlap. Full feature names are provided in a governed key below the panels and in the source tables. Numerical values are not printed inside dense heatmap cells. Separate fully annotated matrices are exported as supplementary figures.


In [ ]:
# Figure source tables.
feature_key = FEATURE_REGISTRY.loc[
    FEATURE_REGISTRY['feature'].isin(PAIRWISE_FEATURES),
    ['feature', 'feature_code', 'family', 'short_label', 'validated_role', 'inference_tier', 'feature_order'],
].sort_values('feature_order')
export_dataframe(feature_key, FIGURE_SOURCE_DIR / 'main_feature_key.csv')
export_dataframe(pairwise_correlations, FIGURE_SOURCE_DIR / 'main_pairwise_structure.csv')
export_dataframe(family_contrast_summary, FIGURE_SOURCE_DIR / 'main_family_contrast.csv')
export_dataframe(pca_components, FIGURE_SOURCE_DIR / 'main_pca_components.csv')
export_dataframe(pca_loadings, FIGURE_SOURCE_DIR / 'main_pca_loadings.csv')

# ------------------------------------------------------------------
# Publication styling and layout
# ------------------------------------------------------------------
mpl.rcParams.update({
    'font.family': ['Arial', 'DejaVu Sans'],
    'font.size': 7.0,
    'axes.titlesize': 8.4,
    'axes.labelsize': 7.2,
    'xtick.labelsize': 6.2,
    'ytick.labelsize': 6.2,
    'legend.fontsize': 6.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

family_order = ['QADD', 'QGAIN', 'QREV', 'QCHAN']
default_colors = mpl.rcParams['axes.prop_cycle'].by_key()['color']
family_colors = {family: default_colors[index] for index, family in enumerate(family_order)}
code_order = [FEATURE_CODES[feature] for feature in feature_order]
code_to_family = feature_lookup.loc[feature_order, 'family'].to_dict()

fig = plt.figure(figsize=(7.35, 11.25), facecolor='white')
outer = fig.add_gridspec(
    4, 2,
    height_ratios=[3.05, 2.25, 2.20, 1.10],
    hspace=0.58,
    wspace=0.42,
)
ax_between = fig.add_subplot(outer[0, 0])
ax_within = fig.add_subplot(outer[0, 1])
ax_scatter = fig.add_subplot(outer[1, 0])
ax_scree = fig.add_subplot(outer[1, 1])
load_grid = outer[2, :].subgridspec(1, 2, wspace=0.36)
ax_load_between = fig.add_subplot(load_grid[0, 0])
ax_load_within = fig.add_subplot(load_grid[0, 1])
ax_key = fig.add_subplot(outer[3, :])

fig.suptitle(
    'Between- and within-participant structure of recording-quality variation',
    y=0.992, fontsize=11.0, fontweight='bold',
)

# Heatmap helper.
def draw_structure_heatmap(axis, matrix, title, letter):
    ordered = matrix.loc[feature_order, feature_order]
    heatmap_cmap = mpl.colormaps['RdBu_r'].copy()
    heatmap_cmap.set_bad('#ececec')
    image = axis.imshow(np.ma.masked_invalid(ordered.to_numpy()), vmin=-1, vmax=1, cmap=heatmap_cmap, aspect='equal')
    axis.set_xticks(np.arange(len(code_order)))
    axis.set_xticklabels(code_order, rotation=90)
    axis.set_yticks(np.arange(len(code_order)))
    axis.set_yticklabels(code_order)
    for label, feature in zip(axis.get_xticklabels(), feature_order):
        label.set_color(family_colors[code_to_family[feature]])
        label.set_fontweight('bold')
    for label, feature in zip(axis.get_yticklabels(), feature_order):
        label.set_color(family_colors[code_to_family[feature]])
        label.set_fontweight('bold')
    boundaries = np.cumsum([5, 4, 4]) - 0.5
    for boundary in boundaries:
        axis.axhline(boundary, color='white', linewidth=1.1)
        axis.axvline(boundary, color='white', linewidth=1.1)
    axis.set_title(f'{letter}  {title}', loc='left', fontweight='bold', pad=6)
    axis.tick_params(length=0)
    return image

image = draw_structure_heatmap(ax_between, between_matrix, 'Between-participant correlations', 'A')
draw_structure_heatmap(ax_within, within_matrix, 'Within-participant correlations', 'B')
# Dedicated colorbar axis prevents collisions with heatmap labels.
heatmap_cax = fig.add_axes([0.915, 0.697, 0.016, 0.205])
colorbar = fig.colorbar(image, cax=heatmap_cax)
colorbar.set_label('Rank correlation', labelpad=5)

# Pair comparison scatter.
scatter_data = pairwise_correlations.loc[
    pairwise_correlations['between_rho'].notna()
    & pairwise_correlations['within_rho'].notna()
].copy()
ax_scatter.axhline(0, color='0.78', linewidth=0.7)
ax_scatter.axvline(0, color='0.78', linewidth=0.7)
ax_scatter.plot([-1, 1], [-1, 1], linestyle='--', color='0.55', linewidth=0.8)

between_family = scatter_data.loc[~scatter_data['same_family'] & ~scatter_data['structural_dependency']]
within_family = scatter_data.loc[scatter_data['same_family'] & ~scatter_data['structural_dependency']]
structural = scatter_data.loc[scatter_data['structural_dependency']]
ax_scatter.scatter(
    between_family['between_rho'], between_family['within_rho'],
    s=15, facecolors='none', edgecolors='0.55', linewidths=0.65,
    label='Between-family pair',
)
ax_scatter.scatter(
    within_family['between_rho'], within_family['within_rho'],
    s=22, color=default_colors[0], alpha=0.75, linewidths=0,
    label='Within-family pair',
)
ax_scatter.scatter(
    structural['between_rho'], structural['within_rho'],
    s=38, marker='D', color=default_colors[1], edgecolor='white', linewidth=0.5,
    label='Governed same-ledger pair', zorder=4,
)
ax_scatter.set_xlim(-1, 1)
ax_scatter.set_ylim(-1, 1)
ax_scatter.set_aspect('equal', adjustable='box')
ax_scatter.set_xlabel('Between-participant correlation')
ax_scatter.set_ylabel('Within-participant correlation')
ax_scatter.set_title('C  Between-versus-within structure', loc='left', fontweight='bold', pad=6)
ax_scatter.legend(frameon=False, loc='lower right')

# Family-organization estimates and uncertainty are reported in the source table
# and supplement rather than printed over the comparison scatter.

# Scree and parallel analysis.
components = np.arange(1, len(PRIMARY_PCA_FEATURES) + 1)
ax_scree.plot(components, 100 * between_eigenvalues / len(PRIMARY_PCA_FEATURES), marker='o', label='Between')
ax_scree.plot(components, 100 * within_eigenvalues / len(PRIMARY_PCA_FEATURES), marker='s', label='Within')
ax_scree.plot(components, 100 * between_parallel95 / len(PRIMARY_PCA_FEATURES), linestyle='--', linewidth=0.9, label='Between parallel 95th')
ax_scree.plot(components, 100 * within_parallel95 / len(PRIMARY_PCA_FEATURES), linestyle=':', linewidth=1.0, label='Within parallel 95th')
ax_scree.set_xticks(components)
ax_scree.set_xlabel('Component')
ax_scree.set_ylabel('Variance explained (%)')
ax_scree.set_title('D  Multilevel PCA: dimensionality', loc='left', fontweight='bold', pad=6)
ax_scree.grid(axis='y', linewidth=0.45, alpha=0.25)
ax_scree.legend(frameon=False, ncol=2, loc='upper right')

# Loading heatmaps show only components meeting the frozen parallel-analysis and
# bootstrap-stability rules. If none qualify, the panel states this explicitly.
def retained_components(level):
    return pca_components.loc[
        (pca_components['level'] == level)
        & pca_components['retained_stable_component'],
        'component',
    ].tolist()

def draw_loading_heatmap(axis, level, title):
    components_to_show = retained_components(level)
    core_codes = [FEATURE_CODES[feature] for feature in PRIMARY_PCA_FEATURES]
    axis.set_title(title, loc='left', fontweight='bold', pad=6)
    if not components_to_show:
        axis.axis('off')
        axis.text(
            0.5, 0.5,
            'No component met both parallel-analysis\nand bootstrap-stability criteria.',
            transform=axis.transAxes, ha='center', va='center', fontsize=7.0,
        )
        return None
    frame = pca_loadings.loc[
        (pca_loadings['level'] == level)
        & pca_loadings['component'].isin(components_to_show)
    ].copy()
    pivot = frame.pivot(index='feature_code', columns='component', values='loading').reindex(core_codes)
    loading_cmap = mpl.colormaps['RdBu_r'].copy()
    loading_cmap.set_bad('#ececec')
    image = axis.imshow(np.ma.masked_invalid(pivot.to_numpy()), vmin=-0.8, vmax=0.8,
                        cmap=loading_cmap, aspect='auto')
    axis.set_xticks(np.arange(len(components_to_show)))
    axis.set_xticklabels([f'PC{component}' for component in components_to_show])
    axis.set_yticks(np.arange(len(core_codes)))
    axis.set_yticklabels(core_codes)
    for label, feature in zip(axis.get_yticklabels(), PRIMARY_PCA_FEATURES):
        label.set_color(family_colors[feature_lookup.loc[feature, 'family']])
        label.set_fontweight('bold')
    axis.tick_params(length=0)
    return image

load_image_between = draw_loading_heatmap(
    ax_load_between, 'between', 'E  Retained between-participant loadings'
)
load_image_within = draw_loading_heatmap(
    ax_load_within, 'within', 'F  Retained within-participant loadings'
)
load_image = load_image_between if load_image_between is not None else load_image_within
if load_image is not None:
    loading_cax = fig.add_axes([0.915, 0.278, 0.016, 0.150])
    load_colorbar = fig.colorbar(load_image, cax=loading_cax)
    load_colorbar.set_label('Eigenvector coefficient', labelpad=5)

# Feature key: four family columns, codes prevent crowding in matrices.
ax_key.axis('off')
ax_key.set_title('Feature key', loc='left', fontweight='bold', pad=2)
for family_index, family in enumerate(family_order):
    family_features = feature_key.loc[feature_key['family'] == family]
    x = 0.02 + family_index * 0.245
    ax_key.text(x, 0.82, family, transform=ax_key.transAxes,
                color=family_colors[family], fontweight='bold', fontsize=7.0)
    lines = [
        f"{row.feature_code}  {row.short_label.replace('†', '')}"
        for row in family_features.itertuples(index=False)
    ]
    ax_key.text(x, 0.68, '\n'.join(lines), transform=ax_key.transAxes,
                va='top', ha='left', fontsize=5.8, linespacing=1.25)

fig.text(
    0.055, 0.020,
    'Between- and within-participant correlations are estimated separately; gray cells indicate insufficient support.\n'
    'Governed same-ledger pairs are identified a priori. PCA is descriptive; no family or global score is formed.',
    fontsize=6.1, ha='left', va='bottom', linespacing=1.25,
)
fig.subplots_adjust(left=0.085, right=0.875, top=0.955, bottom=0.075)

main_paths = save_figure(fig, 'Figure_multidimensional_structure_PUBLICATION', main=True)
main_caption = (
    'Figure X | Between- and within-participant structure of recording-quality variation. '
    'A-B, Family-blocked rank-correlation matrices estimated separately from participant mean profiles '
    '(between participants) and participant-centered repeated recordings with equal participant weight '
    '(within participants). Gray cells did not meet the prespecified joint-support gate. C, Comparison '
    'of pairwise between- and within-participant associations; governed same-ledger pairs are marked '
    'separately and were excluded from the family-organization contrast. D, Between- and within-participant '
    'PCA scree curves with 95th-percentile parallel-analysis thresholds. E-F, Eigenvector coefficients for '
    'components meeting both the parallel-analysis and bootstrap loading-stability criteria. PCA was '
    'descriptive and no component was used to form a family or global quality score.'
)
(FIGURE_MAIN_DIR / 'Figure_multidimensional_structure_PUBLICATION_caption.txt').write_text(
    main_caption, encoding='utf-8'
)
plt.show()
print(main_paths)

# ------------------------------------------------------------------
# Backend-independent deterministic figure-layout audit
# ------------------------------------------------------------------
# Some Jupyter backends expose FigureCanvasBase, which has no
# get_renderer() method. Temporarily attach an Agg canvas for the audit,
# then restore the original notebook canvas before displaying the figure.

from matplotlib.backends.backend_agg import FigureCanvasAgg

original_canvas = fig.canvas

try:
    audit_canvas = FigureCanvasAgg(fig)
    audit_canvas.draw()
    renderer = audit_canvas.get_renderer()

    audit_axes = [
        ("between_heatmap", ax_between),
        ("within_heatmap", ax_within),
        ("comparison_scatter", ax_scatter),
        ("scree", ax_scree),
        ("between_loadings", ax_load_between),
        ("within_loadings", ax_load_within),
        ("heatmap_colorbar", heatmap_cax),
    ]

    if load_image is not None:
        audit_axes.append(("loading_colorbar", loading_cax))

    audit_rows = []
    axis_boxes = {}

    for axis_name, axis in audit_axes:
        bbox = axis.get_window_extent(renderer=renderer)
        axis_boxes[axis_name] = bbox

        audit_rows.append({
            "axis": axis_name,
            "x0_pixels": float(bbox.x0),
            "y0_pixels": float(bbox.y0),
            "x1_pixels": float(bbox.x1),
            "y1_pixels": float(bbox.y1),
            "width_pixels": float(bbox.width),
            "height_pixels": float(bbox.height),
            "positive_width": bool(bbox.width > 0),
            "positive_height": bool(bbox.height > 0),
        })

    # Dedicated colorbars must not intersect any scientific plotting area.
    scientific_names = {
        "between_heatmap",
        "within_heatmap",
        "comparison_scatter",
        "scree",
        "between_loadings",
        "within_loadings",
    }

    collision_rows = []

    for colorbar_name in ["heatmap_colorbar", "loading_colorbar"]:
        if colorbar_name not in axis_boxes:
            continue

        colorbar_bbox = axis_boxes[colorbar_name]

        for scientific_name in scientific_names:
            if scientific_name not in axis_boxes:
                continue

            scientific_bbox = axis_boxes[scientific_name]

            collision_rows.append({
                "colorbar": colorbar_name,
                "panel": scientific_name,
                "intersects": bool(
                    colorbar_bbox.overlaps(scientific_bbox)
                ),
            })

    figure_layout_audit = pd.DataFrame(audit_rows)
    figure_collision_audit = pd.DataFrame(collision_rows)

    export_dataframe(
        figure_layout_audit,
        LOG_DIR / "main_figure_layout_audit.csv",
    )

    export_dataframe(
        figure_collision_audit,
        LOG_DIR / "main_figure_collision_audit.csv",
    )

    if not figure_layout_audit[
        ["positive_width", "positive_height"]
    ].all().all():
        raise AssertionError(
            "Main-figure layout audit found an axis with "
            "non-positive width or height."
        )

    if (
        not figure_collision_audit.empty
        and figure_collision_audit["intersects"].any()
    ):
        failed_collisions = figure_collision_audit.loc[
            figure_collision_audit["intersects"]
        ]

        display(failed_collisions)

        raise AssertionError(
            "Main-figure colorbar collision audit failed. "
            "See main_figure_collision_audit.csv."
        )

    print("Main-figure layout audit: PASS")
    print(
        f"Audited {len(figure_layout_audit)} axes; "
        f"{len(figure_collision_audit)} colorbar-panel comparisons."
    )

finally:
    # Restore the original Jupyter canvas even if an audit assertion fails.
    fig.set_canvas(original_canvas)


# Display only after the audit canvas has been removed.
plt.show()

print("Main figure files:")
for output_name, output_path in main_paths.items():
    print(f"  {output_name}: {output_path}")

display(figure_layout_audit)
display(figure_collision_audit)

## 14. Supplementary figures and full-support visualizations


In [ ]:
# Supplementary Figure 1: pairwise support matrices.
def support_matrix(value_column: str) -> pd.DataFrame:
    matrix = pd.DataFrame(np.nan, index=feature_order, columns=feature_order)
    for row in pair_support.itertuples(index=False):
        value = getattr(row, value_column)
        matrix.loc[row.feature_x, row.feature_y] = value
        matrix.loc[row.feature_y, row.feature_x] = value
    return matrix

between_support_matrix = support_matrix('n_between_participants')
within_support_matrix = support_matrix('n_within_repeated_participants')

fig_support, axes = plt.subplots(1, 2, figsize=(7.2, 3.6), constrained_layout=True)
for axis, matrix, title in [
    (axes[0], between_support_matrix, 'Between: joint participants'),
    (axes[1], within_support_matrix, 'Within: repeated participants'),
]:
    image = axis.imshow(matrix.loc[feature_order, feature_order], cmap='viridis', aspect='equal')
    axis.set_xticks(np.arange(len(code_order)))
    axis.set_xticklabels(code_order, rotation=90)
    axis.set_yticks(np.arange(len(code_order)))
    axis.set_yticklabels(code_order)
    axis.set_title(title, fontweight='bold')
    axis.tick_params(length=0)
    fig_support.colorbar(image, ax=axis, fraction=0.035, pad=0.02)
save_figure(fig_support, 'Supplementary_pairwise_support_matrices', main=False)
plt.show()

# Supplementary Figure 2: annotated separate matrices.
fig_annotated, axes = plt.subplots(1, 2, figsize=(9.0, 4.3), constrained_layout=True)
for axis, matrix, title in [
    (axes[0], between_matrix, 'Between-participant'),
    (axes[1], within_matrix, 'Within-participant'),
]:
    ordered = matrix.loc[feature_order, feature_order]
    annotated_cmap = mpl.colormaps['RdBu_r'].copy()
    annotated_cmap.set_bad('#ececec')
    image = axis.imshow(np.ma.masked_invalid(ordered.to_numpy()), vmin=-1, vmax=1, cmap=annotated_cmap, aspect='equal')
    axis.set_xticks(np.arange(len(code_order)))
    axis.set_xticklabels(code_order, rotation=90)
    axis.set_yticks(np.arange(len(code_order)))
    axis.set_yticklabels(code_order)
    axis.set_title(title, fontweight='bold')
    axis.tick_params(length=0)
    for i in range(len(code_order)):
        for j in range(len(code_order)):
            value = ordered.iloc[i, j]
            if np.isfinite(value) and i != j:
                axis.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=4.5,
                          color='white' if abs(value) > 0.55 else 'black')
fig_annotated.colorbar(image, ax=axes, fraction=0.025, pad=0.02)
save_figure(fig_annotated, 'Supplementary_annotated_correlation_matrices', main=False)
plt.show()

# Supplementary Figure 3: family permutation distributions.
fig_perm, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), constrained_layout=True)
for axis, level in zip(axes, ['between', 'within']):
    null = pd.read_csv(MODEL_DIR / f'family_permutation_{level}.csv')['permuted_delta']
    observed = family_contrast_summary.loc[family_contrast_summary['level'] == level, 'delta'].iloc[0]
    axis.hist(null, bins=35, alpha=0.75)
    axis.axvline(observed, color=default_colors[1], linewidth=1.5)
    axis.set_title(level.capitalize(), fontweight='bold')
    axis.set_xlabel('Permuted family contrast Δ')
    axis.set_ylabel('Count')
save_figure(fig_perm, 'Supplementary_family_permutation', main=False)
plt.show()

# Supplementary Figure 4: PCA bootstrap component stability.
fig_stability, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), constrained_layout=True)
for axis, level in zip(axes, ['between', 'within']):
    subset = pca_components.loc[pca_components['level'] == level]
    axis.errorbar(
        subset['component'],
        subset['bootstrap_loading_similarity_median'],
        yerr=[
            subset['bootstrap_loading_similarity_median'] - subset['bootstrap_loading_similarity_p05'],
            np.zeros(len(subset)),
        ],
        fmt='o', capsize=2,
    )
    axis.axhline(PCA_LOADING_STABILITY_MEDIAN, linestyle='--', color='0.55')
    axis.set_ylim(0, 1.03)
    axis.set_title(level.capitalize(), fontweight='bold')
    axis.set_xlabel('Component')
    axis.set_ylabel('Loading-pattern similarity')
save_figure(fig_stability, 'Supplementary_pca_loading_stability', main=False)
plt.show()


## 15. Final manifest, interpretation guardrails, and completion checks


In [ ]:
interpretation_guardrails = pd.DataFrame([
    {'rule': 'Between and within structures are never pooled.', 'passed': True},
    {'rule': 'Missing pairwise cells are not encoded as zero.', 'passed': True},
    {'rule': 'Known same-ledger relationships are identified before correlations.', 'passed': True},
    {'rule': 'Potential redundancy requires strong structure at both levels and compatible meaning.', 'passed': True},
    {'rule': 'PCA uses no imputation and constructs no quality score.', 'passed': True},
    {'rule': 'QDIST and QTEMP are excluded from structural inference when sparse or invariant.',
     'passed': not structural_eligibility.loc[
         structural_eligibility['family'].isin(['QDIST', 'QTEMP']), 'include_pairwise'
     ].any()},
])
export_dataframe(interpretation_guardrails, LOG_DIR / 'interpretation_guardrails.csv')

output_files = sorted(path for path in OUTPUT_ROOT.rglob('*') if path.is_file())
output_inventory = pd.DataFrame([
    {
        'relative_path': str(path.relative_to(OUTPUT_ROOT)),
        'bytes': path.stat().st_size,
        'sha256': sha256_file(path),
    }
    for path in output_files
])
export_dataframe(output_inventory, MANIFEST_DIR / 'output_inventory.csv')

run_manifest = {
    **FROZEN_SPEC,
    'run_timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scipy': scipy.__version__,
    'pairwise_feature_count': len(PAIRWISE_FEATURES),
    'pair_count': len(pairwise_correlations),
    'primary_pca_features': PRIMARY_PCA_FEATURES,
    'primary_pca_complete_participants': len(between_profiles),
    'primary_pca_within_repeated_participants': within_complete['participant_id'].nunique(),
    'main_figure': main_paths,
    'relationship_class_counts': relationship_summary.set_index('relationship_class')['pair_count'].to_dict(),
}
(MANIFEST_DIR / 'run_manifest.json').write_text(
    json.dumps(run_manifest, indent=2, default=str), encoding='utf-8'
)

completion_checks = pd.DataFrame([
    {'check': 'population audit passed', 'passed': blocking_failures.empty},
    {'check': '17 structural pairwise features audited', 'passed': len(PAIRWISE_FEATURES) == 17},
    {'check': 'all pair support rows exported', 'passed': len(pair_support) == 136},
    {'check': 'correlation estimates exported', 'passed': (TABLE_DIR / 'pairwise_correlations_with_ci.csv').exists()},
    {'check': 'family contrast exported', 'passed': (TABLE_DIR / 'family_contrast_summary.csv').exists()},
    {'check': 'PCA support audit passed', 'passed': pca_support_audit['passed'].all()},
    {'check': 'PCA component stability exported', 'passed': (TABLE_DIR / 'pca_components_and_retention.csv').exists()},
    {'check': 'main PNG exists', 'passed': (FIGURE_MAIN_DIR / 'Figure_multidimensional_structure_PUBLICATION.png').exists()},
    {'check': 'main SVG exists', 'passed': (FIGURE_MAIN_DIR / 'Figure_multidimensional_structure_PUBLICATION.svg').exists()},
    {'check': 'main PDF exists', 'passed': (FIGURE_MAIN_DIR / 'Figure_multidimensional_structure_PUBLICATION.pdf').exists()},
    {'check': 'all interpretation guardrails passed', 'passed': interpretation_guardrails['passed'].all()},
])
export_dataframe(completion_checks, LOG_DIR / 'completion_checks.csv')
display(completion_checks)

if not completion_checks['passed'].all():
    raise AssertionError('One or more final completion checks failed.')

print('Goal 2 multidimensional-structure development run completed successfully.')
if not FINAL_RUN:
    print(
        'This was a development run. Before manuscript export, set FINAL_RUN=True, '
        'restart the kernel, and run all cells to obtain the frozen bootstrap and permutation counts.'
    )
